# Capstone 02 — Multi-agent Research System with Provenance · Practice

**Goal:** build the research system from the [README](README.md) twice, on the Claude Messages API and on the OpenAI Agents SDK, over the same corpus, contracts and eval suite: a coordinator that fans out self-contained briefs in parallel, workers whose evidence comes from platform citations, structured errors with partial results, conflicts kept side by side with sources and dates, and a code-rendered report that passes a provenance audit.

- Brief, requirements R1–R8, architecture and rubric: [README.md](README.md). This notebook holds the code.
- **Estimated time:** 5–7 hours
- **Estimated API cost:** **$0** by default. Everything runs offline on scripted clients. The opt-in live cells in Step 10 cost roughly $0.10–0.30 in total, a planning estimate rather than a measurement (Claude pipeline on `claude-sonnet-5-5` at $2 / $10 per million input / output tokens, one OpenAI `file_search` run on `gpt-6-luna` at $0.10 / $0.50 per million tokens plus $2.50 per 1,000 `file_search` calls, and a vector store that is deleted afterwards; list prices checked 2026-10-04). The live Claude cell prints its token usage so you can compute the real figure. The Agent SDK variant runs Claude Code under your own login or key and is off by default.
- **Prerequisites:** [Claude 03](../../../anthropic-claude/03-agent-sdk/README.md), [Claude 08](../../../anthropic-claude/08-task-decomposition/README.md), [Claude 10](../../../anthropic-claude/10-multi-agent-error-handling/README.md), [Claude 12](../../../anthropic-claude/12-provenance/README.md), [OpenAI 03](../../../openai-codex/03-agents-sdk-and-agents-api/README.md), [OpenAI 10](../../../openai-codex/10-multi-agent-error-handling/README.md), and the repo venv from [00-prerequisites](../../../00-prerequisites/README.md).

| Marker | Meaning |
|---|---|
| **(offline)** | Runs with no key and no network. Claude calls go through `ScriptedClaude`, a real `anthropic.AsyncAnthropic` client whose HTTP layer answers with scripted responses (so the SDK's own types such as `CitationsSearchResultLocation` are parsed for real). OpenAI calls go through the Agents SDK's own test double, `agents.testing.ScriptedModel`. |
| **(live)** | Calls a real API. Guarded by `RUN_LIVE` / `RUN_AGENT_SDK`, both `False` by default. |

The scripted "models" are deliberately simple: they read the request they receive (the brief, the search results, the documents) and answer from it, so they can only cite what was actually passed to them. That makes context-passing bugs visible offline. All sources are **fictional fixtures**.

In [ ]:
import asyncio
import json
import os
import re
import tempfile
import time
from collections import defaultdict
from pathlib import Path
from typing import Any, Callable, Literal

import anthropic
import httpx2
import openai
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field, ValidationError, model_validator

from agents import (Agent, AgentHooks, FileSearchTool, ModelSettings, RunConfig, RunResult, Runner,
                    ToolExecutionConfig, WebSearchTool)
from agents.testing import ModelStep, ScriptedModel, assistant_message, function_call
from openai.types.responses import (ResponseFileSearchToolCall, ResponseFunctionWebSearch,
                                    ResponseOutputMessage, ResponseOutputText)
from openai.types.responses.response_file_search_tool_call import Result as FileSearchResult
from openai.types.responses.response_function_web_search import ActionSearch, ActionSearchSource
from openai.types.responses.response_output_text import AnnotationFileCitation, AnnotationURLCitation

load_dotenv(find_dotenv())
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
AS_OF = "2026-10-03"   # a fixed "today" so every run is reproducible; use date.today() in production

RUN_LIVE = False       # True: run the (live) cells in Step 10 (keys needed; roughly $0.10-0.30, an estimate)
RUN_AGENT_SDK = False  # True: run the Claude Agent SDK variant (starts Claude Code under your login or key)
OFFLINE = RunConfig(tracing_disabled=True)   # offline Agents SDK runs: nothing is exported
print(f"anthropic {anthropic.__version__} | openai {openai.__version__} | Claude: {CLAUDE_MODEL} | OpenAI: {OPENAI_MODEL}")

In [ ]:
# Live cells need keys. Every cell before Step 10 works without them.
if RUN_LIVE:
    assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"
    assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
else:
    print("Offline mode: scripted clients for both stacks. Set RUN_LIVE = True in the setup cell for Step 10.")

## Step 0 — The corpus and the offline toolkit

Ten fictional sources about Riverton's shared e-scooters and e-bikes: six public web sources (`W1`–`W6`) reachable through a search backend, and four attached reports (`D1`–`D4`) for the document analyst. Each sentence that states a finding carries **gold annotations** (facet, metric, value, period). The scripted models use them to answer, and the eval suite uses them to grade. The corpus is built to contain every situation in the brief:

| Situation | Where |
|---|---|
| Two independent counts that agree | `W1` (4.1 M trips) and `D4` (4.0 M trips), 2024 |
| A derivative article (not a second origin) | `W4` re-reports `W1` (`derived_from="W1"`) |
| Credible sources that measure differently | ridership `W1`/`D4` vs `W2` (every unlock counted); injuries `D1` (hospital admissions) vs `D2` (police reports) |
| A genuine, unexplained conflict | CO2 avoided in 2024: `D3` 1,900 t (lifecycle study) vs `W6` 3,400 t (method not stated) |
| A trend that looks like a contradiction | 2.7 M trips (2023) vs 4.1 M trips (2024) |
| A search that times out | any query about revenue or fees (the business-registry backend is down) |
| A search that succeeds with zero results | enforcement fines |

[Theory: Customer brief](README.md#customer-brief)

In [ ]:
QUESTION = ("Prepare a research brief on Riverton's shared e-scooter and e-bike program (launched 2023) for the "
            "council's contract-renewal vote: ridership, safety, emissions, local economy and regulation.")
REQUIRED_FACETS = ["ridership", "safety", "emissions", "economy", "regulation"]


def fact(facet, statement, subject, metric=None, value=None, unit=None, period=None) -> dict:
    return {"facet": facet, "statement": statement, "subject": subject, "metric": metric,
            "value": value, "unit": unit, "period": period}


def source(sid, title, publisher, url, source_type, published, methodology, derived_from, channel, sentences) -> dict:
    return {"source_id": sid, "title": title, "publisher": publisher, "url": url, "source_type": source_type,
            "published": published, "methodology": methodology, "derived_from": derived_from, "channel": channel,
            "sentences": sentences, "text": " ".join(s for s, _ in sentences)}


TRIPS = dict(subject="shared micromobility trips in Riverton", metric="trips per year", unit="million trips")
INJURIES = dict(subject="shared micromobility injuries in Riverton", metric="injuries per million trips", unit="per million trips")
CO2 = dict(subject="CO2 avoided by shared micromobility in Riverton", metric="CO2 avoided", unit="tonnes")

CORPUS = {s["source_id"]: s for s in [
    source("W1", "Shared mobility in 2024: the numbers", "Riverton Transport Authority",
           "https://transport.riverton.example/news/2024-in-numbers", "government", "2025-02-10",
           "validated trip records (trips under 2 minutes excluded)", None, "web", [
               ("Riders took 4.1 million trips on shared e-scooters and e-bikes in 2024.",
                fact("ridership", "Riders took 4.1 million shared e-scooter and e-bike trips in 2024", value=4.1, period="2024", **TRIPS)),
               ("That compares with 2.7 million trips in 2023, the first year of the program.",
                fact("ridership", "Riders took 2.7 million trips in 2023, the program's first year", value=2.7, period="2023", **TRIPS)),
               ("Trips shorter than two minutes are excluded from the count.", None)]),
    source("W2", "Riverton market update", "Micromobility Operators Association",
           "https://operators.example.org/riverton-update", "industry_report", "2025-03-05",
           "operator app data (every unlock counted)", None, "web", [
               ("Operators recorded 5.6 million rides in Riverton in 2024, counting every unlock.",
                fact("ridership", "Operators recorded 5.6 million rides in 2024, counting every unlock", value=5.6, period="2024", **TRIPS)),
               ("The association says short test unlocks explain most of the gap with official counts.", None)]),
    source("W3", "Riverton caps scooter speeds downtown", "Riverton Daily",
           "https://news.riverton.example/speed-cap", "news", "2024-11-18", None, None, "web", [
               ("From 1 December 2024, shared scooters are limited to 20 km/h in the city centre.",
                fact("regulation", "Shared scooters are limited to 20 km/h in the city centre from 1 December 2024",
                     "speed limit for shared scooters in the city centre", "speed limit", 20, "km/h", "2024-12")),
               ("The council said the cap responds to pedestrian complaints.",
                fact("regulation", "The council said the speed cap responds to pedestrian complaints",
                     "reason for the city-centre speed cap", period="2024-11"))]),
    source("W4", "Scooter trips hit a record", "Example City News",
           "https://citynews.example.com/scooter-record", "news", "2025-02-11", None, "W1", "web", [
               ("According to the transport authority, riders took 4.1 million trips in 2024.",
                fact("ridership", "Riders took 4.1 million shared e-scooter and e-bike trips in 2024", value=4.1, period="2024", **TRIPS))]),
    source("W5", "Shared mobility brings local jobs", "Riverton Business Journal",
           "https://business.riverton.example/mobility-jobs", "news", "2024-10-02", None, None, "web", [
               ("The two operators employ about 140 local staff for charging, repairs and rebalancing.",
                fact("economy", "The two operators employ about 140 local staff",
                     "local jobs at the shared-mobility operators", "local staff", 140, "jobs", "2024"))]),
    source("W6", "Our city, cleaner air", "Green Riverton (community newsletter)",
           "https://greenriverton.example.org/cleaner-air", "advocacy", "2025-04-15", None, None, "web", [
               ("Shared scooters and bikes avoided an estimated 3,400 tonnes of CO2 in 2024.",
                fact("emissions", "Shared rides avoided an estimated 3,400 tonnes of CO2 in 2024", value=3400, period="2024", **CO2))]),
    source("D1", "Micromobility injuries 2024: trauma registry report", "Riverton General Hospital",
           None, "academic", "2025-05-20", "hospital emergency admissions", None, "doc", [
               ("In 2024 the registry recorded 38 emergency admissions per million shared-micromobility trips.",
                fact("safety", "The trauma registry recorded 38 emergency admissions per million trips in 2024", value=38, period="2024", **INJURIES)),
               ("Helmet use among admitted riders was 22%.",
                fact("safety", "22% of admitted riders wore a helmet", "helmet use among injured riders", "helmet use", 22, "%", "2024"))]),
    source("D2", "Road Safety Review 2024", "Riverton City Council",
           "https://council.riverton.example/road-safety-review-2024.pdf", "government", "2025-01-28",
           "police-reported collisions", None, "doc", [
               ("Police recorded 21 injury collisions per million shared-micromobility trips in 2024.",
                fact("safety", "Police recorded 21 injury collisions per million trips in 2024", value=21, period="2024", **INJURIES)),
               ("Most collisions happened in the evening peak.", None)]),
    source("D3", "Lifecycle emissions of Riverton's shared fleet", "Example University Transport Lab",
           None, "academic", "2025-06-12", "lifecycle assessment including manufacturing and rebalancing vans", None, "doc", [
               ("Including manufacturing and rebalancing vans, the fleet avoided 1,900 tonnes of CO2 in 2024.",
                fact("emissions", "Including manufacturing and rebalancing vans, the fleet avoided 1,900 tonnes of CO2 in 2024", value=1900, period="2024", **CO2))]),
    source("D4", "Trip count audit 2024", "Regional Mobility Observatory",
           "https://observatory.example.org/riverton-audit-2024", "government", "2025-03-20",
           "independent audit of trip records (trips under 2 minutes excluded)", None, "doc", [
               ("Our audit counted 4.0 million shared-micromobility trips in Riverton in 2024.",
                fact("ridership", "An independent audit counted 4.0 million trips in 2024", value=4.0, period="2024", **TRIPS))]),
]}
DOC_IDS = [s for s in CORPUS if CORPUS[s]["channel"] == "doc"]
URL_TO_ID = {s["url"]: sid for sid, s in CORPUS.items() if s["url"]}

# Gold annotations, looked up by source sentence or by statement (the scripted models and the eval use them)
GOLD: dict[str, dict] = {}
for _s in CORPUS.values():
    for _sentence, _f in _s["sentences"]:
        if _f:
            GOLD[_sentence] = GOLD[_f["statement"]] = _f

for sid, s in CORPUS.items():
    print(f"{sid} [{s['channel']:3}] {s['published']}  {s['publisher'][:38]:38}  method: {s['methodology'] or 'not stated'}"
          + (f"  (derived from {s['derived_from']})" if s["derived_from"] else ""))

In [ ]:
# ---- The search backend: the same index for both stacks, with an injected outage ----
SEARCH_INDEX = {"trips": ["W1", "W2", "W4"], "ridership": ["W1", "W2", "W4"], "emissions": ["W6"],
                "co2": ["W6"], "jobs": ["W5"], "speed": ["W3"], "fines": []}
DOWN = ("revenue", "fees")              # the business-registry backend is down: these queries time out
LATENCY = {"search": 0.10, "model": 0.25}
SEARCH_LOG: list[tuple[str, str]] = []


def lookup(query: str) -> list[dict] | None:
    """None = the backend failed; [] = the search ran and matched nothing."""
    q = query.lower()
    if any(k in q for k in DOWN):
        return None
    ids = list(dict.fromkeys(i for k, v in SEARCH_INDEX.items() if k in q for i in v))
    return [CORPUS[i] for i in ids]


async def search_backend(query: str) -> list[dict]:
    await asyncio.sleep(LATENCY["search"])
    hits = lookup(query)
    SEARCH_LOG.append((query, "timeout" if hits is None else f"{len(hits)} hits"))
    if hits is None:
        raise TimeoutError(f"search backend did not answer within 8 s: {query!r}")
    return hits


print(lookup("Riverton shared scooter trips 2024") and [s["source_id"] for s in lookup("Riverton shared scooter trips 2024")])
print("fines ->", lookup("Riverton scooter enforcement fines 2024"), "| revenue ->", lookup("Riverton micromobility tax revenue"))

### Offline toolkit (run once, no need to study it)

`ScriptedClaude` wraps a real `AsyncAnthropic` client. Its transport routes each request to a **responder** by what the request contains (the `delegate` tool means the coordinator, the `search_web` tool means a web researcher, `output_config` means the field-extraction call, `document` blocks mean the document analyst, anything else is the synthesizer), waits `LATENCY["model"]` seconds, and answers. Responders are pure functions of the request, so concurrent calls are safe. A responder can also return `"TIMEOUT"` to simulate a read timeout. Every request body is kept in `.requests`.

In [ ]:
_counter = iter(range(1, 10**9))


def _id(prefix: str) -> str:
    return f"{prefix}_{next(_counter):05d}"


def text_block(t: str, citations: list[dict] | None = None) -> dict:
    return {"type": "text", "text": t, **({"citations": citations} if citations else {})}


def tool_use_block(name: str, tool_input: dict) -> dict:
    return {"type": "tool_use", "id": _id("toolu"), "name": name, "input": tool_input}


def reply(*blocks: dict) -> dict:
    stop = "tool_use" if any(b["type"] == "tool_use" for b in blocks) else "end_turn"
    return {"id": _id("msg"), "type": "message", "role": "assistant", "model": "scripted", "content": list(blocks),
            "stop_reason": stop, "stop_sequence": None, "usage": {"input_tokens": 0, "output_tokens": 0}}


def route(body: dict) -> str:
    tools = {t.get("name") for t in body.get("tools", [])}
    if "delegate" in tools:
        return "coordinator"
    if "search_web" in tools:
        return "web_researcher"
    if "output_config" in body:
        return "fields"
    first = body["messages"][0]["content"]
    if isinstance(first, list) and any(b.get("type") == "document" for b in first):
        return "document_analyst"
    return "synthesizer"


class ScriptedClaude:
    """Offline Messages API: a real AsyncAnthropic client whose HTTP layer calls scripted responders."""

    def __init__(self, responders: dict[str, Callable[[dict], dict | str]], latency: float | None = None):
        self.responders, self.requests = responders, []
        self.latency = LATENCY["model"] if latency is None else latency
        transport = httpx2.MockTransport(self._handle)
        self.client = anthropic.AsyncAnthropic(api_key="offline", max_retries=0,
                                               http_client=anthropic.DefaultAsyncHttpxClient(transport=transport))

    async def _handle(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append(body)
        await asyncio.sleep(self.latency)
        role = route(body)
        if role not in self.responders:
            return httpx2.Response(500, json={"type": "error", "error": {"type": "api_error", "message": f"no responder for {role}"}})
        out = self.responders[role](body)
        if out == "TIMEOUT":
            raise httpx2.ReadTimeout("scripted read timeout", request=request)
        return httpx2.Response(200, json=out)


def tag(text: str, name: str) -> str:
    m = re.search(rf"<{name}>(.*?)</{name}>", text, re.S)
    return m.group(1).strip() if m else ""


def split_sentences(text: str) -> list[str]:
    return [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", text.strip()) if s]


print("toolkit ready")

## Step 1 — Data contracts (R4, R5)

Everything that moves between agents is a typed record. A `Claim` keeps **content** (statement, subject, metric, value, unit, data period) apart from **metadata** (evidence with source, verbatim excerpt and location; the source registry holds title, publisher, URL, publication date, method and lineage). A `SubagentResult` is the envelope every subagent run ends in, success or not, and its validator rejects the contradictory envelopes that hide failures. [Theory: R4](README.md#r4-claim-records-that-separate-content-from-metadata) · [R5](README.md#r5-structured-error-propagation)

In [ ]:
Facet = Literal["ridership", "safety", "emissions", "economy", "regulation"]
FailureType = Literal["timeout", "rate_limited", "unavailable", "invalid_input", "permission_denied", "internal_error"]


class Source(BaseModel):
    source_id: str
    title: str
    publisher: str
    url: str | None                 # null for internal or attached documents without a URL
    source_type: str
    published: str | None           # publication date; null when unknown, never guessed
    methodology: str | None
    derived_from: str | None        # the origin this source only re-reports


REGISTRY = {sid: Source(**{k: v for k, v in s.items() if k in Source.model_fields}) for sid, s in CORPUS.items()}


class Evidence(BaseModel):
    source_id: str
    excerpt: str                    # what the citation points at
    location: str | None            # "chars 0-71", "block 0", "answer chars 0-58", "file D1.txt"
    verbatim: bool                  # True: quoted from the source; False: a span of the model's own answer


class Claim(BaseModel):
    claim_id: str
    facet: Facet
    statement: str                  # keeps the source's qualifiers ("estimated", "preliminary")
    subject: str
    metric: str | None
    value: float | None
    unit: str | None
    period: str | None              # the DATA period the value describes, not the publication date
    evidence: list[Evidence] = Field(min_length=1)    # no evidence, no claim


class SubagentResult(BaseModel):
    task_id: str
    subagent: Literal["web_researcher", "document_analyst"]
    facets: list[Facet]
    status: Literal["success", "empty", "partial", "failed"]
    failure_type: FailureType | None = None
    retryable: bool | None = None
    attempted: list[str] = Field(default_factory=list)       # every query or document tried
    failed_queries: list[str] = Field(default_factory=list)  # tried, but the source did not answer
    empty_queries: list[str] = Field(default_factory=list)   # answered with zero matches (a finding)
    claims: list[Claim] = Field(default_factory=list)        # partial results survive a failure
    alternatives: list[str] = Field(default_factory=list)    # "query: ..." or free-text suggestions
    coverage_impact: str | None = None
    message: str | None = None

    @model_validator(mode="after")
    def consistent(self):
        failing = self.status in ("partial", "failed")
        if failing and (self.failure_type is None or self.retryable is None):
            raise ValueError("partial/failed results must say what failed and whether a retry can help")
        if not failing and self.failure_type is not None:
            raise ValueError("success/empty results must not carry a failure_type")
        if self.status == "partial" and not self.claims:
            raise ValueError("partial means some findings survived; use 'failed' otherwise")
        if self.status == "empty" and self.claims:
            raise ValueError("an empty result cannot contain claims")
        return self


class ClaimFields(BaseModel):       # what the structured-output step returns for one cited statement
    ref: str
    facet: Facet
    subject: str
    metric: str | None
    value: float | None
    unit: str | None
    period: str | None


class ClaimFieldsBatch(BaseModel):
    claims: list[ClaimFields]


class Brief(BaseModel):             # what the coordinator fills in for one subagent
    task_id: str
    subagent: Literal["web_researcher", "document_analyst"]
    facets: list[Facet]
    objective: str
    queries: list[str]              # suggested searches (web researcher)
    documents: list[str]            # attached report IDs (document analyst)


for bad in [dict(status="failed"), dict(status="partial", failure_type="timeout", retryable=True),
            dict(status="success", failure_type="timeout")]:
    try:
        SubagentResult(task_id="T9", subagent="web_researcher", facets=["economy"], **bad)
    except ValidationError as exc:
        print(f"rejected {bad}: {exc.errors()[0]['msg']}")

> **Observe:** the validator turns the three exam anti-patterns into errors: a bare `failed` with no failure type is the generic "search unavailable"; a `partial` with nothing in it hides a total failure; a `success` that carries a failure type is a timeout dressed up as success.

## Step 2 — The plan and the briefs (R1, R2)

The coordinator decides the plan; **code** decides whether the plan is acceptable before anything runs. `check_plan()` rejects a plan that leaves a required facet unassigned (the over-narrow decomposition), duplicates work, or sends a subagent off without inputs. `render_brief()` is the only way a subagent learns anything: goal, facets, what is out of scope, suggested inputs, prior findings verbatim, what already failed, today's date and the output contract. Both stacks use this one function. [Theory: R1](README.md#r1-coordinator-and-scoped-subagents) · [R2](README.md#r2-explicit-context-passing)

In [ ]:
class PlanError(ValueError):
    pass


def check_plan(briefs: list[Brief], required: list[str] = REQUIRED_FACETS, max_tasks: int = 6) -> None:
    if not briefs:
        raise PlanError("the coordinator delegated nothing")
    if len(briefs) > max_tasks:
        raise PlanError(f"{len(briefs)} tasks exceed the cap of {max_tasks}")
    if len({b.task_id for b in briefs}) != len(briefs):
        raise PlanError("duplicate task IDs")
    missing = [f for f in required if f not in {f for b in briefs for f in b.facets}]
    if missing:
        raise PlanError(f"plan leaves required facets unassigned: {missing}")
    owner: dict[tuple, str] = {}
    for b in briefs:
        if b.subagent == "web_researcher" and not b.queries:
            raise PlanError(f"{b.task_id}: a web task needs suggested queries")
        if b.subagent == "document_analyst" and (not b.documents or set(b.documents) - set(DOC_IDS)):
            raise PlanError(f"{b.task_id}: document tasks may only use attached reports {DOC_IDS}")
        for f in b.facets:
            if (b.subagent, f) in owner:
                raise PlanError(f"facet {f!r} assigned twice to {b.subagent}: {owner[(b.subagent, f)]} and {b.task_id}")
            owner[(b.subagent, f)] = b.task_id


PLAN = [
    {"task_id": "T1", "subagent": "web_researcher", "facets": ["ridership", "emissions"],
     "objective": "Find published 2023-2024 trip counts and emissions estimates for the shared fleet.",
     "queries": ["Riverton shared scooter trips 2024", "Riverton micromobility emissions CO2"], "documents": []},
    {"task_id": "T2", "subagent": "web_researcher", "facets": ["economy"],
     "objective": "Find evidence on local jobs and public revenue linked to the program.",
     "queries": ["Riverton shared mobility local jobs", "Riverton micromobility tax revenue"], "documents": []},
    {"task_id": "T3", "subagent": "web_researcher", "facets": ["regulation"],
     "objective": "Find the current rules for shared scooters and how they are enforced.",
     "queries": ["Riverton scooter speed limit city centre", "Riverton scooter enforcement fines 2024"], "documents": []},
    {"task_id": "T4", "subagent": "document_analyst", "facets": ["safety", "emissions", "ridership"],
     "objective": "Extract injury rates, emissions estimates and trip counts from the attached reports.",
     "queries": [], "documents": DOC_IDS},
]
check_plan([Brief(**t) for t in PLAN])
print("full plan accepted")

narrow = [Brief(**t) for t in PLAN if t["task_id"] in ("T1", "T4")]     # what a hurried coordinator might do
try:
    check_plan(narrow)
except PlanError as exc:
    print("narrow plan rejected:", exc)

In [ ]:
CONTRACT = {
    "web_researcher": ("Search with search_web; run independent queries in the same turn. Answer only from the results "
                       "you receive: one short sentence per finding, and let the citation feature attach the source. "
                       "Keep qualifiers such as 'estimated' or 'preliminary'. State no number that is not in a cited "
                       "passage. If a search finds nothing, say so; if a search fails, say which query failed."),
    "document_analyst": ("Read only the attached reports listed in <documents>. One short sentence per finding, cited. "
                         "Keep qualifiers. If two reports disagree, report both values; do not choose between them."),
}


def render_brief(brief: Brief, prior: list[Claim] = (), failed: list[str] = ()) -> str:
    """The ONLY channel from coordinator to subagent: everything it needs, nothing it should not do."""
    others = [f for f in REQUIRED_FACETS if f not in brief.facets]
    prior_json = json.dumps([c.model_dump() for c in prior], ensure_ascii=False) if prior else "none"
    return f"""<task id="{brief.task_id}" subagent="{brief.subagent}">
<goal>{QUESTION}</goal>
<as_of>{AS_OF}</as_of>
<your_facets>{", ".join(brief.facets)}</your_facets>
<objective>{brief.objective}</objective>
<out_of_scope>{", ".join(others) or "none"} (other subagents cover these: do not research them)</out_of_scope>
<suggested_queries>{json.dumps(brief.queries)}</suggested_queries>
<documents>{json.dumps(brief.documents)}</documents>
<prior_findings>{prior_json}</prior_findings>
<already_failed>{json.dumps(list(failed))}</already_failed>
<output_contract>{CONTRACT[brief.subagent]}</output_contract>
</task>"""


def brief_from_text(text: str) -> Brief:
    """Recover the Brief from a rendered brief (used by the OpenAI extractor, which only sees the nested run)."""
    head = re.search(r'<task id="([^"]+)" subagent="([^"]+)">', text)
    return Brief(task_id=head.group(1), subagent=head.group(2),
                 facets=[f.strip() for f in tag(text, "your_facets").split(",") if f.strip()],
                 objective=tag(text, "objective"), queries=json.loads(tag(text, "suggested_queries") or "[]"),
                 documents=json.loads(tag(text, "documents") or "[]"))


WEAK_BRIEF = "Look into the economy side of the scooter thing and report back."
REQUIRED_PARTS = ["<goal>", "<as_of>", "<your_facets>", "<out_of_scope>", "<prior_findings>", "<already_failed>", "<output_contract>"]


def brief_lint(text: str) -> list[str]:
    return [p for p in REQUIRED_PARTS if p not in text]


print("weak brief is missing:", brief_lint(WEAK_BRIEF))
print("strong brief is missing:", brief_lint(render_brief(Brief(**PLAN[1]))))
print(render_brief(Brief(**PLAN[1]))[:900])

> **Observe:** the weak brief would work in a single-agent chat, where the model can see the earlier conversation. A subagent sees nothing but its prompt, so "the scooter thing" is meaningless to it. The strong brief also names the facets that are **out of scope**, which is how you prevent two subagents from doing the same work.

> **Try it:** remove `"economy"` from `T2` in `PLAN` and run `check_plan` again. Then move `"emissions"` from `T1` to `T3` and see which check fires.

## Step 3 — Coordinator and parallel fan-out on Claude (R1, R3)

The coordinator is one Messages API call with a single `delegate` tool. Its system prompt states the goal and quality criteria, not a procedure. It answers with **several `delegate` `tool_use` blocks in one response**; code checks the plan, runs one worker per block with `asyncio.gather` under a semaphore, and sends **all** `tool_result` blocks back in **one** user message. The workers themselves are built in Step 4; this step wires them up and measures the latency gain. [Theory: R3](README.md#r3-parallel-fan-out) · [Stack A](README.md#stack-a-claude-messages-api)

In [ ]:
COORDINATOR_SYSTEM = f"""You coordinate a research team for the City of Riverton's policy unit. Today is {AS_OF}.
Goal: a research brief that analysts can defend line by line in a council session.
Quality criteria:
- Every required facet is covered: {", ".join(REQUIRED_FACETS)}.
- Each task has a distinct scope: split by facet and by source type. No two tasks of the same subagent type share a facet.
- web_researcher searches public web sources; document_analyst reads only the attached reports.
- Independent tasks are delegated in the SAME response so they run in parallel.
- A subagent sees nothing but its brief: make every brief self-contained."""

DELEGATE_TOOL = {
    "name": "delegate",
    "description": ("Start one subagent on one scoped research task. The subagent sees ONLY what you put here plus the "
                    "attached reports you list. Call it several times in the same response for independent tasks. "
                    "Returns a structured result: status, claims found, failures and coverage impact."),
    "input_schema": {
        "type": "object",
        "properties": {
            "task_id": {"type": "string", "description": "Short unique ID, e.g. T1"},
            "subagent": {"type": "string", "enum": ["web_researcher", "document_analyst"]},
            "facets": {"type": "array", "items": {"type": "string", "enum": REQUIRED_FACETS}},
            "objective": {"type": "string", "description": "What to find, in one or two sentences"},
            "queries": {"type": "array", "items": {"type": "string"}, "description": "Suggested searches (web_researcher)"},
            "documents": {"type": "array", "items": {"type": "string", "enum": DOC_IDS}, "description": "Attached report IDs (document_analyst)"},
        },
        "required": ["task_id", "subagent", "facets", "objective", "queries", "documents"],
    },
}


def coordinator_responder(body: dict) -> dict:
    """Scripted coordinator: first turn delegates the four tasks in ONE response; second turn acknowledges."""
    last = body["messages"][-1]
    if isinstance(last["content"], list) and any(b.get("type") == "tool_result" for b in last["content"]):
        return reply(text_block("All delegations returned. Proceed to synthesis; keep the coverage gaps visible."))
    return reply(text_block("Delegating four independent tasks in parallel."), *[tool_use_block("delegate", t) for t in PLAN])


print(json.dumps(DELEGATE_TOOL["input_schema"]["properties"]["facets"]))

## Step 4 — Workers with Citations (R4)

**Web researcher:** a bounded tool loop with one client tool, `search_web`. The tool returns `search_result` blocks with citations enabled (one text block per sentence, so a citation points at exactly one sentence), or a single text block `"No results found."` for a valid empty result. A timeout is retried **once inside the tool**; if it persists, the tool returns an `is_error` result with structured JSON. The loop has a budget of four tool turns and must end on `end_turn`: a worker that runs out of turns returns a `partial` or `failed` envelope naming the queries whose results it never read, never a quiet `empty`, and any other stop reason (`max_tokens`, `refusal`) is an error.

**Document analyst:** one call with the attached reports as `document` blocks, citations enabled on **all** of them, and the source ID and publication date in each block's `context` (passed to the model, never cited).

**Citations to claims, in code:** `search_result_location` gives the search result's `source` (our URL) and the cited block; `char_location` gives `document_index` and character offsets. Neither the URL nor the quote is ever retyped by the model. A second call, `messages.parse(output_format=ClaimFieldsBatch)`, adds metric, value, unit and period from the cited text, because citations and structured outputs cannot share one request. [Theory: R4](README.md#r4-claim-records-that-separate-content-from-metadata) · [Claude 12 §12.7](../../../anthropic-claude/12-provenance/README.md#127-the-citations-api-document-blocks-expanded-beyond-the-guide)

In [ ]:
SEARCH_TOOL = {
    "name": "search_web",
    "description": ("Search the public web index for sources about Riverton. One query per call; call it several times in "
                    "one turn for independent queries. Returns citable search results, the text 'No results found.' when "
                    "the search ran and matched nothing, or an error object when the search backend failed."),
    "input_schema": {"type": "object", "properties": {"query": {"type": "string", "description": "Keyword query"}},
                     "required": ["query"]},
}
ALTERNATIVE_QUERY = {"Riverton micromobility tax revenue": "Riverton micromobility permit fees"}   # specialist knowledge
USAGE: list[tuple[str, int, int]] = []    # (role, input tokens, output tokens) for every model call


def track(role: str, response) -> None:
    USAGE.append((role, response.usage.input_tokens, response.usage.output_tokens))


def as_search_results(sources: list[dict]) -> list[dict]:
    """Tool output. If any block in a tool_result is a search_result, all of them must be."""
    if not sources:
        return [{"type": "text", "text": "No results found."}]
    return [{"type": "search_result", "source": s["url"],
             "title": f"{s['title']} ({s['publisher']}, published {s['published']})",
             "content": [{"type": "text", "text": t} for t, _ in s["sentences"]],
             "citations": {"enabled": True}} for s in sources]


async def search_with_recovery(query: str, retries: int = 1) -> tuple[list[dict] | None, dict | None]:
    """Local recovery: retry a timeout inside the tool, propagate only what cannot be fixed here."""
    last = ""
    for _ in range(retries + 1):
        try:
            return await search_backend(query), None
        except TimeoutError as exc:
            last = str(exc)
    return None, {"error": "search_failed", "failure_type": "timeout", "query": query,
                  "attempts": retries + 1, "retryable": True, "message": last}


def block_span(first: int, end_exclusive: int) -> str:
    last = end_exclusive - 1
    return f"block {first}" if last == first else f"blocks {first}-{last}"


def drafts_from_citations(response, doc_ids: list[str]) -> list[dict]:
    """One draft claim per cited text block; evidence straight from the citation objects."""
    drafts = []
    for b in response.content:
        if b.type != "text" or not b.citations:
            continue
        evidence = []
        for c in b.citations:
            if c.type == "search_result_location":
                evidence.append(Evidence(source_id=URL_TO_ID.get(c.source, c.source), excerpt=c.cited_text,
                                         location=block_span(c.start_block_index, c.end_block_index), verbatim=True))
            elif c.type == "char_location":
                evidence.append(Evidence(source_id=doc_ids[c.document_index], excerpt=c.cited_text,
                                         location=f"chars {c.start_char_index}-{c.end_char_index}", verbatim=True))
        if evidence:
            drafts.append({"statement": b.text.strip().rstrip("."), "evidence": evidence})
    return drafts


FIELDS_SYSTEM = ("You turn cited statements into structured fields. Use only the statement and its excerpt. "
                 "period is the time the value describes, not the publication date; use null when it is not stated. "
                 "Keep units as written (normalize 'rides' and 'trips' to 'million trips' only when the excerpt says million).")


async def structure_claude(ac, brief: Brief, drafts: list[dict]) -> list[Claim]:
    """Step two of the two-step design: structured fields over the ALREADY cited text."""
    if not drafts:
        return []
    items = [{"ref": f"d{i}", "statement": d["statement"], "excerpt": d["evidence"][0].excerpt} for i, d in enumerate(drafts)]
    r = await ac.messages.parse(model=CLAUDE_MODEL, max_tokens=2000, system=FIELDS_SYSTEM, output_format=ClaimFieldsBatch,
                                messages=[{"role": "user", "content": f"Allowed facets: {brief.facets}\n<items>{json.dumps(items)}</items>"}])
    track("fields", r)
    if r.stop_reason != "end_turn" or r.parsed_output is None:
        raise ValueError(f"field extraction stopped with {r.stop_reason}")
    fields = {f.ref: f for f in r.parsed_output.claims}
    return [Claim(claim_id=f"{brief.task_id}-{i}", statement=d["statement"], evidence=d["evidence"],
                  **fields[f"d{i}"].model_dump(exclude={"ref"}))
            for i, d in enumerate(drafts) if f"d{i}" in fields]


def build_envelope(brief: Brief, claims: list[Claim], log: list[tuple[str, str]]) -> SubagentResult:
    """Status is computed from the tool log in code, never taken from the model's impression."""
    failed = [q for q, outcome in log if outcome not in ("hits", "empty")]
    common = dict(task_id=brief.task_id, subagent=brief.subagent, facets=brief.facets, claims=claims,
                  attempted=[q for q, _ in log], empty_queries=[q for q, o in log if o == "empty"])
    if not failed:
        return SubagentResult(status="success" if claims else "empty", **common)
    outcome = next(o for q, o in log if q == failed[0])
    alternatives = [f"query: {ALTERNATIVE_QUERY[q]}" for q in failed if q in ALTERNATIVE_QUERY
                    and ALTERNATIVE_QUERY[q] not in common["attempted"]]
    alternatives.append("Ask an analyst to request the figures from the city finance office")
    return SubagentResult(status="partial" if claims else "failed", failure_type=outcome,
                          retryable=outcome in ("timeout", "unavailable", "rate_limited"), failed_queries=failed,
                          alternatives=alternatives, message=f"{len(failed)} of {len(log)} searches did not answer",
                          coverage_impact=f"Could not search: {', '.join(failed)}. Findings on {', '.join(brief.facets)} may be incomplete.",
                          **common)

In [ ]:
WEB_SYSTEM = ("You are a web researcher subagent. Work only on the task in the user message, using the search_web tool. "
              "Answer only from search results you received in this conversation.")
DOC_SYSTEM = ("You are a document analyst subagent. Work only on the task in the user message and only from the "
              "attached reports. Write one short sentence per finding; the citation feature attaches the source.")


MAX_TOOL_TURNS = 4


async def run_web_researcher(ac, brief: Brief, prior=(), failed=()) -> SubagentResult:
    messages = [{"role": "user", "content": render_brief(brief, prior, failed)}]
    log: list[tuple[str, str]] = []
    responses, unread = [], []
    for _ in range(MAX_TOOL_TURNS):                            # bounded tool loop
        response = await ac.messages.create(model=CLAUDE_MODEL, max_tokens=1500, system=WEB_SYSTEM,
                                            tools=[SEARCH_TOOL], messages=messages)
        track("web_researcher", response)
        responses.append(response)
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            break
        calls = [b for b in response.content if b.type == "tool_use"]
        outcomes = await asyncio.gather(*(search_with_recovery(c.input["query"]) for c in calls))   # parallel tool calls
        results = []
        for call, (sources, error) in zip(calls, outcomes):
            if error:
                log.append((call.input["query"], error["failure_type"]))
                results.append({"type": "tool_result", "tool_use_id": call.id, "content": json.dumps(error), "is_error": True})
            else:
                log.append((call.input["query"], "hits" if sources else "empty"))
                results.append({"type": "tool_result", "tool_use_id": call.id, "content": as_search_results(sources)})
        messages.append({"role": "user", "content": results})  # ALL results in ONE message
        unread = [c.input["query"] for c in calls]             # no model call has read these results yet
    else:                                                      # never reached end_turn: the budget ran out
        claims = await structure_claude(ac, brief, [d for r in responses for d in drafts_from_citations(r, [])])
        return budget_exhausted(brief, claims, log, unread)
    if response.stop_reason != "end_turn":                     # max_tokens, refusal, ...: not a finished answer
        raise RuntimeError(f"web researcher stopped with {response.stop_reason}")
    claims = await structure_claude(ac, brief, [d for r in responses for d in drafts_from_citations(r, [])])
    return build_envelope(brief, claims, log)


def budget_exhausted(brief: Brief, claims: list[Claim], log: list[tuple[str, str]], unread: list[str]) -> SubagentResult:
    """The tool-turn budget ran out: a failure with what was found so far, never a quiet 'empty'."""
    return SubagentResult(
        task_id=brief.task_id, subagent=brief.subagent, facets=brief.facets, claims=claims,
        status="partial" if claims else "failed", failure_type="internal_error", retryable=False,
        attempted=[q for q, _ in log], failed_queries=unread,
        empty_queries=[q for q, o in log if o == "empty" and q not in unread],
        alternatives=["Re-run with a larger tool-turn budget or fewer suggested queries"],
        message=f"tool-turn budget of {MAX_TOOL_TURNS} exhausted before the worker answered",
        coverage_impact=f"Results of {', '.join(unread)} were never read. Findings on {', '.join(brief.facets)} may be incomplete.")


async def run_document_analyst(ac, brief: Brief, prior=(), failed=()) -> SubagentResult:
    docs = [CORPUS[d] for d in brief.documents]
    content = [{"type": "document", "source": {"type": "text", "media_type": "text/plain", "data": d["text"]},
                "title": d["title"], "citations": {"enabled": True},          # all documents, or none
                "context": json.dumps({"source_id": d["source_id"], "publisher": d["publisher"], "published": d["published"]})}
               for d in docs]
    content.append({"type": "text", "text": render_brief(brief, prior, failed)})
    response = await ac.messages.create(model=CLAUDE_MODEL, max_tokens=2000, system=DOC_SYSTEM,
                                        messages=[{"role": "user", "content": content}])
    track("document_analyst", response)
    if response.stop_reason != "end_turn":
        raise RuntimeError(f"document analyst stopped with {response.stop_reason}")
    drafts = drafts_from_citations(response, [d["source_id"] for d in docs])
    cited = {e.source_id for d in drafts for e in d["evidence"]}
    log = [(f"document {d}", "hits" if d in cited else "empty") for d in brief.documents]
    return build_envelope(brief, await structure_claude(ac, brief, drafts), log)

In [ ]:
# ---- Scripted responders for the workers: they answer ONLY from what the request contains ----
def web_responder(body: dict) -> dict:
    msgs = body["messages"]
    brief_text = msgs[0]["content"] if isinstance(msgs[0]["content"], str) else msgs[0]["content"][0]["text"]
    if len(msgs) == 1:                                                     # turn 1: search every suggested query at once
        return reply(*[tool_use_block("search_web", {"query": q}) for q in json.loads(tag(brief_text, "suggested_queries"))])
    blocks, result_index = [], 0                                           # turn 2: cite what came back
    for m in msgs:
        if m["role"] != "user" or isinstance(m["content"], str):
            continue
        for tr in m["content"]:
            if tr.get("type") != "tool_result" or not isinstance(tr.get("content"), list):
                continue
            for sr in tr["content"]:
                if sr.get("type") != "search_result":
                    continue
                for bi, b in enumerate(sr["content"]):
                    if b["text"] in GOLD:
                        blocks.append(text_block(GOLD[b["text"]]["statement"] + ". ", [{
                            "type": "search_result_location", "cited_text": b["text"], "source": sr["source"],
                            "title": sr["title"], "search_result_index": result_index,
                            "start_block_index": bi, "end_block_index": bi + 1}]))
                result_index += 1
    return reply(*(blocks or [text_block("No sources found for the assigned queries.")]))


def doc_responder(body: dict) -> dict:
    docs = [b for b in body["messages"][0]["content"] if b.get("type") == "document"]
    blocks = []
    for di, d in enumerate(docs):
        data = d["source"]["data"]
        for sentence in split_sentences(data):
            if sentence in GOLD:
                start = data.index(sentence)
                blocks.append(text_block(GOLD[sentence]["statement"] + ". ", [{
                    "type": "char_location", "cited_text": sentence, "document_index": di, "document_title": d["title"],
                    "start_char_index": start, "end_char_index": start + len(sentence)}]))
    return reply(*blocks)


def fields_from_gold(items: list[dict]) -> dict:
    out = []
    for it in items:
        f = GOLD.get(it.get("excerpt", "")) or GOLD.get(it["statement"])
        if f:
            out.append({"ref": it["ref"], **{k: f[k] for k in ("facet", "subject", "metric", "value", "unit", "period")}})
    return {"claims": out}


def fields_responder(body: dict) -> dict:
    return reply(text_block(json.dumps(fields_from_gold(json.loads(tag(body["messages"][0]["content"], "items"))))))


RESPONDERS = {"coordinator": coordinator_responder, "web_researcher": web_responder,
              "document_analyst": doc_responder, "fields": fields_responder}

sc = ScriptedClaude(RESPONDERS)
t4 = await run_document_analyst(sc.client, Brief(**PLAN[3]))
print(t4.status, "|", len(t4.claims), "claims |", t4.attempted)
for c in t4.claims[:3]:
    e = c.evidence[0]
    print(f"  {c.claim_id}: {c.statement} | {c.metric}={c.value} {c.unit} ({c.period}) <- {e.source_id} {e.location}: {e.excerpt!r}")
sent = sc.requests[-1]
print("structured-output request carries output_config:", "output_config" in sent, "| citations in it:", "citations" in json.dumps(sent))

In [ ]:
# Integrity check: every char_location excerpt must be exactly the slice of the source text it points at.
def integrity_problems(claims: list[Claim]) -> list[str]:
    problems = []
    for c in claims:
        for e in c.evidence:
            text = CORPUS.get(e.source_id, {}).get("text", "")
            if e.location and e.location.startswith("chars "):
                a, b = map(int, e.location[6:].split("-"))
                if text[a:b].strip() != e.excerpt.strip():
                    problems.append(f"{c.claim_id}: offsets {a}-{b} do not match the excerpt")
            elif e.verbatim and e.excerpt not in text:
                problems.append(f"{c.claim_id}: excerpt not found in {e.source_id}")
    return problems


print("document analyst integrity:", integrity_problems(t4.claims) or "OK")

> **Observe:** the second request (field extraction) carries `output_config` and **no** citations, and the first carried citations and no `output_config`. Putting both in one request returns a 400 on the real API. The claim's evidence (source ID, verbatim excerpt, character offsets) comes from the first call and is never touched by the second.

> **Try it:** change `doc_responder` so it cites a sentence that is **not** in the document (for example append `" (approx.)"` to `cited_text`) and rerun the integrity check. In a prompt-only pipeline this kind of drift goes unnoticed.

## Step 5 — Error propagation (R5)

The economy task hits the outage. Watch the layers: the **tool** retries the timeout once (local recovery), then returns an `is_error` result with structured JSON; the **worker wrapper** computes a `partial` envelope that keeps the jobs finding, lists what was attempted and what failed, and suggests an alternative query; the **coordinator's policy** decides to retry once with that alternative (passing the partial findings and the failed queries in the new brief), and when that also fails, accepts the partial result and lets the report annotate the gap. The regulation task shows the other side: a search that **succeeded with zero results** is an `empty_queries` entry, not a failure. [Theory: R5](README.md#r5-structured-error-propagation)

In [ ]:
def failed_envelope(brief: Brief, failure_type: str, retryable: bool, message: str) -> SubagentResult:
    return SubagentResult(task_id=brief.task_id, subagent=brief.subagent, facets=brief.facets, status="failed",
                          failure_type=failure_type, retryable=retryable, attempted=brief.queries or brief.documents,
                          failed_queries=brief.queries or brief.documents, message=message,
                          alternatives=[f"query: {ALTERNATIVE_QUERY[q]}" for q in brief.queries if q in ALTERNATIVE_QUERY],
                          coverage_impact=f"No findings for {', '.join(brief.facets)} from this task.")


async def run_worker(ac, brief: Brief, prior=(), failed=()) -> SubagentResult:
    """Every outcome becomes an envelope; nothing escapes to cancel sibling workers."""
    fn = run_web_researcher if brief.subagent == "web_researcher" else run_document_analyst
    try:
        return await fn(ac, brief, prior, failed)
    except anthropic.APITimeoutError as exc:                       # after the SDK's own retries
        return failed_envelope(brief, "timeout", True, f"model call timed out: {exc}")
    except anthropic.RateLimitError as exc:
        return failed_envelope(brief, "rate_limited", True, str(exc))
    except anthropic.APIStatusError as exc:
        if exc.status_code in (401, 403):
            return failed_envelope(brief, "permission_denied", False, str(exc))
        return failed_envelope(brief, "internal_error", exc.status_code >= 500, str(exc))
    except Exception as exc:  # noqa: BLE001 - connection errors, bad model output, bugs: none may cancel siblings
        return failed_envelope(brief, "internal_error", False, f"{type(exc).__name__}: {exc}")


SEARCH_LOG.clear()
t2 = await run_worker(sc.client, Brief(**PLAN[1]))
print(t2.model_dump_json(indent=1, exclude={"claims"}))
print("partial claims kept:", [c.statement for c in t2.claims])
print("search log:", SEARCH_LOG)
t3 = await run_worker(sc.client, Brief(**PLAN[2]))
print("\nT3:", t3.status, "| empty_queries:", t3.empty_queries, "| claims:", len(t3.claims))

In [ ]:
# A model-call timeout (not a tool timeout): the worker cannot report it itself, so the wrapper does.
broken = ScriptedClaude({**RESPONDERS, "web_researcher": lambda body: "TIMEOUT"})
print(await run_worker(broken.client, Brief(**PLAN[2])))

In [ ]:
Action = Literal["accept", "retry_modified", "accept_partial", "escalate", "annotate_gap"]


def decide(r: SubagentResult, retries_left: int) -> Action:
    if r.status in ("success", "empty"):
        return "accept"                                  # zero results is an answer
    if r.failure_type == "permission_denied":
        return "escalate"                                # a retry cannot grant access
    if r.retryable and retries_left > 0 and any(a.startswith("query: ") for a in r.alternatives):
        return "retry_modified"                          # follow the specialist's suggestion, never the same query
    if r.status == "partial":
        return "accept_partial"
    return "annotate_gap"


def merge_envelopes(old: SubagentResult, new: SubagentResult) -> SubagentResult:
    still_failing = new.status in ("partial", "failed")
    claims = old.claims + new.claims
    return old.model_copy(update={
        "status": ("partial" if claims else "failed") if still_failing else ("success" if claims else "empty"),
        "failure_type": new.failure_type if still_failing else None,
        "retryable": new.retryable if still_failing else None,
        "attempted": old.attempted + new.attempted,
        "failed_queries": old.failed_queries + new.failed_queries if still_failing else [],
        "empty_queries": old.empty_queries + new.empty_queries, "claims": claims,
        "alternatives": [a for a in new.alternatives if not a.startswith("query: ")] if still_failing else [],
        "coverage_impact": " ".join(x for x in (old.coverage_impact, new.coverage_impact) if x) if still_failing else None,
        "message": new.message})


async def recover(ac, briefs: list[Brief], results: list[SubagentResult], retries: int = 1):
    final, decisions = list(results), []
    for i, b in enumerate(briefs):
        left = retries
        while (action := decide(final[i], left)) == "retry_modified":
            alt = next(a for a in final[i].alternatives if a.startswith("query: "))[len("query: "):]
            retry = b.model_copy(update={"task_id": f"{b.task_id}-R{retries - left + 1}", "queries": [alt]})
            new = await run_worker(ac, retry, prior=final[i].claims, failed=final[i].failed_queries)   # explicit context
            decisions.append({"task": b.task_id, "action": "retry_modified", "retry": retry.task_id, "query": alt, "outcome": new.status})
            final[i], left = merge_envelopes(final[i], new), left - 1
        decisions.append({"task": b.task_id, "action": action, "status": final[i].status})
    return final, decisions


(t2_final,), d = await recover(sc.client, [Brief(**PLAN[1])], [t2])
for x in d:
    print(x)
print(t2_final.status, t2_final.failure_type, "| failed:", t2_final.failed_queries, "| claims:", len(t2_final.claims))
retry_brief = next(r for r in sc.requests if route(r) == "web_researcher" and "T2-R1" in json.dumps(r))
print("retry brief carries prior findings:", "140 local staff" in json.dumps(retry_brief),
      "| and the failed query:", "tax revenue" in tag(retry_brief["messages"][0]["content"], "already_failed"))

> **Observe:** three different things are true about economy at the end, and the envelope keeps them apart: one finding survived (140 local staff), two queries never answered (and which ones), and nothing suggests the outage will clear soon. The coordinator did not retry the same query; it followed the specialist's alternative once and then stopped. The retry brief carried the partial findings and the failed queries, so the subagent knew what not to repeat.

> **Try it:** set `retries=0` in `recover` and compare the decisions. Then remove `"fees"` from `DOWN` so the alternative query succeeds: the merged status becomes `success`, and the report no longer has a gap for economy.

### The full Claude run: coordinator → parallel workers → recovery → one message of results

Now the pieces run together. First a fair timing comparison of the same plan run sequentially and in parallel (same scripted latencies), then the complete coordinator run.

In [ ]:
async def fan_out(ac, briefs: list[Brief], limit: int = 4) -> list[SubagentResult]:
    gate = asyncio.Semaphore(limit)

    async def one(b: Brief) -> SubagentResult:
        async with gate:
            return await run_worker(ac, b)

    return list(await asyncio.gather(*(one(b) for b in briefs)))    # gather, not TaskGroup: siblings survive


async def run_sequential(ac, briefs: list[Brief]) -> list[SubagentResult]:
    return [await run_worker(ac, b) for b in briefs]


briefs = [Brief(**t) for t in PLAN]
t0 = time.perf_counter(); seq = await run_sequential(sc.client, briefs); t_seq = time.perf_counter() - t0
t0 = time.perf_counter(); par = await fan_out(sc.client, briefs); t_par = time.perf_counter() - t0
assert [r.model_dump() for r in seq] == [r.model_dump() for r in par], "same plan, same results"
print(f"sequential {t_seq:.2f}s | parallel {t_par:.2f}s | speed-up x{t_seq / t_par:.1f}")

In [ ]:
def summarize_for_coordinator(r: SubagentResult) -> str:
    """What goes back in the tool_result: the envelope minus the claim bodies (claims are passed by reference)."""
    return json.dumps({**r.model_dump(exclude={"claims"}), "claim_ids": [c.claim_id for c in r.claims]})


async def coordinate_claude(ac, *, retries: int = 1) -> dict:
    t0 = time.perf_counter()
    messages = [{"role": "user", "content": f"{QUESTION}\nAttached reports: "
                 + json.dumps({d: CORPUS[d]["title"] for d in DOC_IDS})}]
    r1 = await ac.messages.create(model=CLAUDE_MODEL, max_tokens=2000, system=COORDINATOR_SYSTEM,
                                  tools=[DELEGATE_TOOL], messages=messages)
    track("coordinator", r1)
    calls = [b for b in r1.content if b.type == "tool_use" and b.name == "delegate"]
    briefs = [Brief.model_validate(c.input) for c in calls]
    check_plan(briefs)                                                  # R1: reject a narrow plan before spending
    results = await fan_out(ac, briefs)                                 # R3: one response -> concurrent workers
    results, decisions = await recover(ac, briefs, results, retries)    # R5: explicit, bounded recovery policy
    messages += [{"role": "assistant", "content": r1.content},
                 {"role": "user", "content": [{"type": "tool_result", "tool_use_id": c.id, "content": summarize_for_coordinator(r)}
                                              for c, r in zip(calls, results)]}]   # ALL results, ONE message
    r2 = await ac.messages.create(model=CLAUDE_MODEL, max_tokens=500, system=COORDINATOR_SYSTEM,
                                  tools=[DELEGATE_TOOL], messages=messages)
    track("coordinator", r2)
    note = "".join(b.text for b in r2.content if b.type == "text")
    return {"briefs": briefs, "results": results, "decisions": decisions, "note": note,
            "delegations_in_first_response": len(calls), "seconds": time.perf_counter() - t0}


sc = ScriptedClaude(RESPONDERS)
claude_run = await coordinate_claude(sc.client)
print("delegations in the first coordinator response:", claude_run["delegations_in_first_response"])
for r in claude_run["results"]:
    print(f"  {r.task_id} {r.subagent:16} {r.status:8} claims={len(r.claims)} failed={r.failed_queries} empty={r.empty_queries}")
last_coord = [r for r in sc.requests if route(r) == "coordinator"][-1]["messages"][-1]
print("tool_result blocks in the coordinator's final user message:", sum(b["type"] == "tool_result" for b in last_coord["content"]))
print("coordinator:", claude_run["note"], f"| {claude_run['seconds']:.2f}s")

> **Observe:** the four workers started together because the four `tool_use` blocks arrived in one response; the run takes about as long as the slowest worker plus the economy retry. The coordinator gets compact envelopes (status, failures, claim IDs) in **one** user message; the claim bodies stay in code and go to synthesis by reference, which keeps the coordinator's context small without a lossy summarizing hop.

## Step 6 — Conflicts and dates (R7)

Code classifies every pair of numeric claims about the same subject, metric and unit, **in this order**: different periods are a trend (`temporal_difference`), a missing period is `undetermined`, values within 5% `agree`, known and different methods are a `methodology_difference`, and anything else is a genuine `conflict`. Methods are compared by **independent origin**: `W4` inherits `W1`'s method because it only re-reports it. Conflict records carry both values, sources, publication dates, data periods and methods, and the coordinator's reconciliation policy decides what the report does with them. Nobody picks a winner. [Theory: R7](README.md#r7-conflict-handling-with-both-sources-and-dates) · [Claude 12 §12.2](../../../anthropic-claude/12-provenance/README.md#122-handling-conflicting-data)

In [ ]:
def origins(c: Claim) -> set[str]:
    return {(REGISTRY[e.source_id].derived_from if e.source_id in REGISTRY else None) or e.source_id for e in c.evidence}


def methods_of(c: Claim) -> set[str | None]:
    return {REGISTRY[o].methodology if o in REGISTRY else None for o in origins(c)}


def same_quantity(a: Claim, b: Claim) -> bool:
    return (a.subject.lower(), (a.metric or "").lower(), (a.unit or "").lower()) == \
           (b.subject.lower(), (b.metric or "").lower(), (b.unit or "").lower())


def classify_pair(a: Claim, b: Claim) -> str:
    if not same_quantity(a, b):
        return "different_quantity"
    if a.value is None or b.value is None:
        return "not_numeric"
    if a.period is None or b.period is None:
        return "undetermined"
    if a.period != b.period:
        return "temporal_difference"
    if abs(a.value - b.value) <= 0.05 * max(abs(a.value), abs(b.value)):
        return "agree"
    ma, mb = methods_of(a), methods_of(b)
    if None not in ma | mb and ma != mb:
        return "methodology_difference"
    return "conflict"


def classify_all(claims: list[Claim]) -> list[tuple[str, str, str]]:
    out = []
    for i, a in enumerate(claims):
        for b in claims[i + 1:]:
            kind = classify_pair(a, b)
            if kind not in ("different_quantity", "not_numeric"):
                out.append((a.claim_id, b.claim_id, kind))
    return out


def side(c: Claim) -> dict:
    o = sorted(origins(c))
    return {"claim_id": c.claim_id, "value": c.value, "unit": c.unit, "period": c.period, "sources": o,
            "titles": [REGISTRY[s].title for s in o if s in REGISTRY],
            "published": [REGISTRY[s].published or "date unknown" for s in o if s in REGISTRY],
            "methods": [REGISTRY[s].methodology or "method not stated" for s in o if s in REGISTRY],
            "excerpts": [e.excerpt for e in c.evidence]}


def conflict_records(claims: list[Claim], pairs) -> list[dict]:
    """One record per quantity and period, holding every claim involved in a disagreement about it."""
    by_id = {c.claim_id: c for c in claims}
    groups: dict[tuple, dict] = {}
    for a, b, kind in pairs:
        if kind not in ("conflict", "methodology_difference"):
            continue
        c = by_id[a]
        g = groups.setdefault((c.subject, c.metric, c.unit, c.period), {"ids": set(), "kinds": set()})
        g["ids"] |= {a, b}
        g["kinds"].add(kind)
    return [{"claims": sorted(g["ids"], key=lambda x: int(x[1:])),
             "kind": "conflict" if "conflict" in g["kinds"] else "methodology_difference",
             "subject": subject, "metric": metric, "period": period,
             "sides": [side(by_id[x]) for x in sorted(g["ids"], key=lambda x: int(x[1:]))],
             "status": "unresolved", "reconciliation": None}
            for (subject, metric, unit, period), g in groups.items()]


def reconcile(records: list[dict]) -> list[dict]:
    """The coordinator's reconciliation policy. It never selects a value; it decides how both are presented."""
    for r in records:
        if r["kind"] == "methodology_difference":
            r["reconciliation"] = "present both with their methods; they measure different things"
            r["status"] = "explained"
        else:
            r["reconciliation"] = "present both as contested; request verification from the authors before the vote"
    return records

In [ ]:
def merge_claims(results: list[SubagentResult]) -> list[Claim]:
    """Merge, never replace: same value for the same thing -> one claim, union of evidence. Stable IDs C1..Cn."""
    order = {f: i for i, f in enumerate(REQUIRED_FACETS)}
    merged: dict[tuple, Claim] = {}
    for r in results:
        for c in r.claims:
            k = (c.subject.lower(), (c.metric or "").lower(), (c.unit or "").lower(), c.period, c.value,
                 c.statement if c.value is None else None)
            if k in merged:
                seen = {(e.source_id, e.excerpt) for e in merged[k].evidence}
                merged[k].evidence += [e for e in c.evidence if (e.source_id, e.excerpt) not in seen]
            else:
                merged[k] = c.model_copy(deep=True)
    ordered = sorted(merged.values(), key=lambda c: (order[c.facet], c.subject, c.period or "", -(c.value or 0), c.statement))
    return [c.model_copy(update={"claim_id": f"C{i}"}) for i, c in enumerate(ordered, 1)]


claims = merge_claims(claude_run["results"])
pairs = classify_all(claims)
records = reconcile(conflict_records(claims, pairs))
for c in claims:
    print(f"{c.claim_id:4} {c.facet:10} {c.value!s:>6} {c.unit or '':18} {c.period or '':8} origins={sorted(origins(c))}")
print()
for a, b, kind in pairs:
    print(f"{a}/{b}: {kind}")

> **Observe:** `W1` and `W4` state the same 4.1 million and were merged into one claim with two evidence items but **one** origin. 4.1 (W1) and 4.0 (D4) `agree`. Both differ from 5.6 (W2) by method, which is explained, not contested. 2.7 (2023) vs the 2024 counts is a trend. Only CO2 avoided is a genuine conflict: same quantity, same year, and one side does not state a method. If you checked agreement **before** periods, two equal values for 2023 and 2024 would "agree" and be merged, and a time series would vanish.

> **Try it:** give `W6` a methodology (`REGISTRY["W6"].methodology = "operator estimate"`), rerun the classification and watch the CO2 pair change from `conflict` to `methodology_difference`. Restore it afterwards (`= None`).

## Step 7 — Synthesis, rendering and audit (R6, R8)

Support levels come from independent origins (`well_established` needs two that agree; a derivative article adds nothing), never from model confidence. The coverage table is computed from the plan and the envelopes. The synthesizer gets **all** records verbatim and writes only summary sentences that end in `[C#]` or `[gap:facet]` references. Code renders every number (key figures, contested findings, trends, coverage, sources), and the audit blocks a draft that cites nothing, cites an unknown claim, contains a number that is not in its evidence, states one side of a conflict or of a methodology difference, or skips a gap. [Theory: R6](README.md#r6-coverage-annotations) · [R8](README.md#r8-attribution-that-survives-synthesis)

In [ ]:
def support_levels(claims: list[Claim], pairs) -> dict[str, str]:
    parent = {c.claim_id: c.claim_id for c in claims}

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for a, b, kind in pairs:
        if kind == "agree":
            parent[find(a)] = find(b)
    clusters = defaultdict(list)
    for c in claims:
        clusters[find(c.claim_id)].append(c)
    contested = {x for a, b, kind in pairs if kind == "conflict" for x in (a, b)}
    levels = {}
    for c in claims:
        independent = set().union(*(origins(x) for x in clusters[find(c.claim_id)]))
        levels[c.claim_id] = ("contested" if c.claim_id in contested
                              else "well_established" if len(independent) >= 2 else "single_source")
    return levels


def coverage_rows(briefs: list[Brief], results: list[SubagentResult]) -> list[dict]:
    rows = []
    for facet in REQUIRED_FACETS:
        rs = [r for r in results if facet in r.facets]
        answered = sorted({c.evidence[0].source_id for r in rs for c in r.claims if c.facet == facet})
        failed = [q for r in rs for q in r.failed_queries]
        empty = [q for r in rs for q in r.empty_queries]
        if not rs:
            label, reason = "NONE", "never assigned to a subagent"
        elif failed:
            label = "PARTIAL" if answered else "NONE"
            kinds = sorted({r.failure_type for r in rs if r.failure_type})
            reason = f"{'/'.join(kinds)} on: " + "; ".join(f"'{q}'" for q in failed)
        else:
            label, reason = "FULL", ""
        rows.append({"facet": facet, "label": label, "reason": reason, "answered": answered,
                     "empty": empty, "failed": failed})
    return rows


levels = support_levels(claims, pairs)
rows = coverage_rows(claude_run["briefs"], claude_run["results"])
print({k: v for k, v in levels.items()})
for r in rows:
    print(r)

In [ ]:
SYNTH_SYSTEM = """You write the summary section of a research brief from the records in the user message.
Rules:
- 6 to 12 sentences. Every sentence ends with the IDs of the claims it uses, e.g. [C3][C5], or with a gap tag [gap:<facet>].
- Use only claims in <claims>; copy numbers exactly; keep qualifiers such as 'estimated'.
- For a pair marked conflict or methodology_difference in <pairs>, give both values in the same sentence with both IDs.
- For every facet whose coverage label is not FULL, write one sentence with its gap tag saying what is missing and why.
- A failed source is a gap, never 'no evidence'. A search with zero results is a finding."""


def synthesis_prompt(claims, pairs, levels, rows) -> str:
    return (f"<goal>{QUESTION}</goal>\n<as_of>{AS_OF}</as_of>\n"
            f"<claims>{json.dumps([c.model_dump() for c in claims], ensure_ascii=False)}</claims>\n"
            f"<pairs>{json.dumps([{'a': a, 'b': b, 'kind': k} for a, b, k in pairs])}</pairs>\n"
            f"<support>{json.dumps(levels)}</support>\n"
            f"<coverage>{json.dumps([{k: r[k] for k in ('facet', 'label', 'reason')} for r in rows])}</coverage>")


def fmt(v: float) -> str:
    """3400 -> '3,400', 38 -> '38', 4.0 -> '4.0', 4.1 -> '4.1' (the way the sources write them)."""
    if v != int(v):
        return f"{v:g}"
    return f"{v:,.0f}" if abs(v) >= 1000 else (f"{v:.0f}" if abs(v) >= 10 else f"{v:.1f}")


def agree_cluster(cid: str, pairs) -> set[str]:
    """Claims connected to cid through 'agree' pairs (including cid)."""
    cluster, frontier = {cid}, [cid]
    while frontier:
        x = frontier.pop()
        for a, b, kind in pairs:
            if kind == "agree" and x in (a, b):
                y = b if x == a else a
                if y not in cluster:
                    cluster.add(y)
                    frontier.append(y)
    return cluster


def scripted_summary(prompt: str) -> str:
    """Stand-in for the synthesizer model: it can only use what the prompt passed to it."""
    cl = {c["claim_id"]: c for c in json.loads(tag(prompt, "claims"))}
    prs = json.loads(tag(prompt, "pairs"))
    tuples = [(p["a"], p["b"], p["kind"]) for p in prs]
    groups = defaultdict(list)
    for c in cl.values():
        if c["value"] is not None:
            groups[(c["subject"], c["metric"], c["unit"], c["period"])].append(c["claim_id"])
    sentences = []
    for (subject, metric, unit, period), ids in groups.items():
        kinds = {p["kind"] for p in prs if p["a"] in ids and p["b"] in ids}
        vals = " versus ".join(f"{fmt(cl[i]['value'])} {unit} [{i}]" for i in ids)
        agreeing = max((agree_cluster(i, tuples) & set(ids) for i in ids), key=len)
        if "conflict" in kinds:
            sentences.append(f"Sources disagree on {metric} in {period}: {vals}; see Contested findings.")
        elif len(agreeing) > 1:
            both = " and ".join(f"{fmt(cl[i]['value'])} {unit} [{i}]" for i in ids if i in agreeing)
            rest = " and ".join(f"{fmt(cl[i]['value'])} {unit} [{i}]" for i in ids if i not in agreeing)
            sentences.append(f"Independent sources agree on {metric} in {period}: {both}"
                             + (f"; a figure measured differently is {rest}." if rest else "."))
        elif "methodology_difference" in kinds:
            sentences.append(f"Figures for {metric} in {period} depend on the counting method: {vals}.")
        elif len(ids) > 1:
            sentences.append(f"Independent sources agree on {metric} in {period}: {vals}.")
        else:
            sentences.append(f"{cl[ids[0]]['statement']} [{ids[0]}].")
    sentences += [f"{c['statement']} [{c['claim_id']}]." for c in cl.values() if c["value"] is None]
    sentences += [f"Findings on {r['facet']} are {r['label'].lower()}: a source did not answer, so they may be incomplete [gap:{r['facet']}]."
                  for r in json.loads(tag(prompt, "coverage")) if r["label"] != "FULL"]
    return " ".join(sentences)


def synth_responder(body: dict) -> dict:
    return reply(text_block(scripted_summary(body["messages"][0]["content"])))


RESPONDERS["synthesizer"] = synth_responder


async def synthesize_claude(ac, claims, pairs, levels, rows) -> str:
    r = await ac.messages.create(model=CLAUDE_MODEL, max_tokens=1500, system=SYNTH_SYSTEM,
                                 messages=[{"role": "user", "content": synthesis_prompt(claims, pairs, levels, rows)}])
    track("synthesizer", r)
    if r.stop_reason != "end_turn":
        raise RuntimeError(f"synthesis stopped with {r.stop_reason}; do not ship a truncated summary")
    return "".join(b.text for b in r.content if b.type == "text")


summary = await synthesize_claude(sc.client, claims, pairs, levels, rows)
for s in split_sentences(summary):
    print("-", s)

In [ ]:
def render_report(stack: str, claims, pairs, levels, rows, records, summary, *, lossy: bool = False) -> str:
    by_id = {c.claim_id: c for c in claims}
    out = [f"# Riverton shared micromobility: research brief", f"_As of {AS_OF} · built with: {stack}_", "",
           "## Summary", "", summary, ""]
    if not lossy:
        out += ["## Well-established findings", ""]
        shown: set[str] = set()
        for c in claims:
            if levels[c.claim_id] == "well_established" and c.claim_id not in shown:
                cluster = sorted(agree_cluster(c.claim_id, pairs), key=lambda x: int(x[1:]))
                shown |= set(cluster)
                ind = sorted(set().union(*(origins(by_id[x]) for x in cluster)))
                out.append(f"- {c.metric}, {c.period}: " + " and ".join(f"{fmt(by_id[x].value)} {by_id[x].unit} [{x}]" for x in cluster)
                           + f" agree (independent origins: {', '.join(ind)})")
        out.append("")
        out += ["## Contested findings", ""]
        for r in records:
            out += [f"**{r['metric']}, {r['period']}** ({r['kind'].replace('_', ' ')}; {r['status']}). Coordinator: {r['reconciliation']}.", "",
                    "| Claim | Value | Period | Sources (published) | Method |", "|---|---|---|---|---|"]
            for s in r["sides"]:
                srcs = "; ".join(f"[{sid}] {t} ({p})" for sid, t, p in zip(s["sources"], s["titles"], s["published"]))
                out.append(f"| {s['claim_id']} | {fmt(s['value'])} {s['unit']} | {s['period']} | {srcs} | {'; '.join(s['methods'])} |")
            out.append("")
        series = defaultdict(list)
        for c in claims:
            if c.value is not None:
                series[(c.subject, c.metric, c.unit)].append(c)
        trends = [v for v in series.values() if len({c.period for c in v}) > 1]
        if trends:
            out += ["## Trends over time", ""]
            for v in trends:
                periods = sorted({c.period for c in v if c.period})
                out.append(f"- {v[0].metric}: " + " → ".join(
                    f"{p}: " + ", ".join(f"{fmt(c.value)} {c.unit} [{c.claim_id}]" for c in v if c.period == p) for p in periods)
                    + " (figures measured differently are compared in Contested findings)")
            out.append("")
    numeric = sorted((c for c in claims if c.value is not None), key=lambda c: (REQUIRED_FACETS.index(c.facet), c.metric or "", c.period or ""))
    out += ["## Key figures", "", "| Facet | Metric | Value | Period | Sources (published) | Method | Support |", "|---|---|---|---|---|---|---|"]
    for c in numeric:
        o = sorted(origins(c))
        out.append(f"| {c.facet} | {c.metric} | {fmt(c.value)} {c.unit} | {c.period or 'unknown'} | "
                   + "; ".join(f"[{s}] ({REGISTRY[s].published or 'date unknown'})" for s in o if s in REGISTRY)
                   + f" | {'; '.join(REGISTRY[s].methodology or 'not stated' for s in o if s in REGISTRY)} | {levels[c.claim_id].replace('_', ' ')} [{c.claim_id}] |")
    out.append("")
    if not lossy:
        out += ["## Coverage and limitations", "", "| Facet | Coverage | Answered by | Searched, no sources found | Unavailable |", "|---|---|---|---|---|"]
        for r in rows:
            out.append(f"| {r['facet']} | {r['label']}{' (' + r['reason'] + ')' if r['reason'] else ''} | {', '.join(r['answered']) or '-'} | "
                       f"{'; '.join(r['empty']) or '-'} | {'; '.join(r['failed']) or '-'} |")
        single = [c.claim_id for c in claims if levels[c.claim_id] == "single_source"]
        out += ["", f"- Single-source findings: {', '.join(single) or 'none'}.",
                "- A facet marked PARTIAL or NONE had a source that did not answer: findings there may be incomplete. "
                "'Searched, no sources found' means the search ran and matched nothing.", ""]
    cited = sorted({s for c in claims for s in origins(c) | {e.source_id for e in c.evidence}}) if not lossy else \
        sorted({e.source_id for cid in re.findall(r"\[(C\d+)\]", summary) if cid in by_id for e in by_id[cid].evidence})
    out += ["## Sources", ""]
    for sid in cited:
        s = REGISTRY.get(sid)
        if s:
            out.append(f"- [{sid}] {s.title}. {s.publisher}, published {s.published or 'date unknown'}. "
                       f"{s.url or 'Attached report'}. Method: {s.methodology or 'not stated'}"
                       + (f". Re-reports [{s.derived_from}]" if s.derived_from else "") + ".")
    return "\n".join(out)


report_claude = render_report("Claude Messages API", claims, pairs, levels, rows, records, summary)
display(Markdown(report_claude))

In [ ]:
REF = re.compile(r"\[(C\d+|gap:[a-z]+)\]")
NUM = re.compile(r"(?<![A-Za-z\d.,])\d+(?:[.,]\d+)*")


def nums(text: str) -> set[float]:
    return {float(n.replace(",", "")) for n in NUM.findall(text)}


def evidence_numbers(c: Claim) -> set[float]:
    """Numbers the claim's CITED evidence supports (never the whole source document).

    Verbatim evidence: the numbers in the cited excerpt. Non-verbatim evidence (an answer span plus a URL): fetch the
    source and take the numbers of the sentences that state the claim's value. Plus the claim's data period.
    """
    out = nums(c.period or "")
    for e in c.evidence:
        if e.verbatim:
            out |= nums(e.excerpt)
        elif c.value is not None:
            text = CORPUS.get(e.source_id, {}).get("text", "")
            out |= set().union(*(nums(s) for s in split_sentences(text) if c.value in nums(s)))
    return out


def audit(summary: str, claims: list[Claim], pairs, rows) -> list[str]:
    by_id, problems, gaps_seen = {c.claim_id: c for c in claims}, [], set()
    not_full = {r["facet"] for r in rows if r["label"] != "FULL"}
    paired = [(a, b, k) for a, b, k in pairs if k in ("conflict", "methodology_difference")]   # both sides, always
    for s in split_sentences(summary):
        refs = REF.findall(s)
        if not refs:
            problems.append(f"no reference: {s[:70]!r}")
            continue
        ids = [r for r in refs if r.startswith("C")]
        for g in (r[4:] for r in refs if r.startswith("gap:")):
            gaps_seen.add(g)
            if g not in not_full:
                problems.append(f"gap tag for a fully covered facet {g!r}")
        unknown = [r for r in ids if r not in by_id]
        if unknown:
            problems.append(f"unknown claim IDs {unknown}: {s[:60]!r}")
            continue
        if ids:
            allowed = set().union(*(evidence_numbers(by_id[r]) for r in ids))
            stray = nums(REF.sub("", s)) - allowed
            if stray:
                problems.append(f"numbers not in the cited evidence {sorted(stray)}: {s[:60]!r}")
        for a, b, kind in paired:
            if (a in ids) != (b in ids):
                problems.append(f"states one side of {kind.replace('_', ' ')} {a}/{b}: {s[:60]!r}")
        if re.search(r"\bno evidence\b", s, re.I):
            problems.append(f"'no evidence' wording: {s[:60]!r}")
    problems += [f"no gap sentence for facet {f!r}" for f in sorted(not_full - gaps_seen)]
    return problems


print("audit (Claude build):", audit(summary, claims, pairs, rows) or "PASS")

> **Observe:** the model never typed a number into a table. The key figures, contested values, trends and coverage are rendered from records, so they cannot drift. The only free text is the summary, and the audit checks every number in it against the evidence of the claims it cites.

> **Try it:** edit one summary sentence, e.g. `summary.replace("1,900", "1,800")`, and rerun `audit`. Then delete the economy gap sentence and run it again.

## Step 8 — The OpenAI build: agents as tools (R1–R8)

Same corpus, same contracts, same downstream code. Only the model-facing layer changes:

- A **manager** agent calls `research_web` and `analyze_documents`, created with `Agent.as_tool(parameters=Brief, input_builder=..., custom_output_extractor=..., failure_error_function=...)`. The `input_builder` renders the structured `Brief` with the same `render_brief()`.
- **The plan is checked before delegation.** `as_tool` calls run as soon as the manager emits them, so a `PlanGate` agent hook runs the same `check_plan()` in `on_llm_end`, which the SDK calls right after the model returns and before it executes the tool calls. A narrow plan raises `PlanError` with no specialist started, as on Claude.
- The **web researcher** has `WebSearchTool()`. Its answer carries `url_citation` annotations: a URL and a span **of the model's answer**, not a quote from the page, so evidence is marked `verbatim=False`.
- The **document analyst** has `FileSearchTool(include_search_results=True)`. Its answer carries `file_citation` annotations (a file and an index), and the included search results give the chunk text, used as verbatim evidence.
- The **extractor** turns the nested run's items into the same `SubagentResult` JSON (status from the search calls' `status` and `queries`: a failed or incomplete `file_search_call` is a gap, a completed one with no results is a valid empty result), and a separate structuring agent with `output_type=ClaimFieldsBatch` adds the fields. Citations that share one insertion `index` are grouped into one claim, and each one's evidence is the retrieved chunk that best supports the statement, not just the last chunk from that file.

Offline, `ScriptedModel` responders play the models: they read the brief from the call, simulate the hosted searches against the same index and outage, and emit real `web_search_call` / `file_search_call` items and annotated messages. [Theory: Stack B](README.md#stack-b-openai-agents-sdk) · [OpenAI 03 §3.4](../../../openai-codex/03-agents-sdk-and-agents-api/README.md#34-handoffs-vs-agents-as-tools)

In [ ]:
FILE_IDS = {d: f"file_{d}" for d in DOC_IDS}             # what a vector-store upload would return
FILE_TO_ID = {v: k for k, v in FILE_IDS.items()}


def brief_text_of(call) -> str:
    return call.input if isinstance(call.input, str) else call.input[-1]["content"]


async def oa_web_responder(call):
    """Scripted web researcher: hosted searches against the same index, answer with url_citation annotations."""
    await asyncio.sleep(LATENCY["model"])
    text = brief_text_of(call)
    task = brief_from_text(text).task_id
    items, answer, annotations = [], "", []
    for i, q in enumerate(json.loads(tag(text, "suggested_queries"))):
        hits = lookup(q)
        items.append(ResponseFunctionWebSearch(
            id=f"ws_{task}_{i}", type="web_search_call", status="failed" if hits is None else "completed",
            action=ActionSearch(type="search", query=q, sources=[ActionSearchSource(type="url", url=s["url"]) for s in hits or []])))
        for s in hits or []:
            for sentence, f in s["sentences"]:
                if f:
                    start = len(answer)
                    answer += f["statement"] + ". "
                    annotations.append(AnnotationURLCitation(type="url_citation", url=s["url"], title=s["title"],
                                                             start_index=start, end_index=len(answer) - 1))
    msg = ResponseOutputMessage(id=f"msg_{task}", type="message", role="assistant", status="completed",
                                content=[ResponseOutputText(type="output_text", annotations=annotations,
                                                            text=answer or "No sources found for the assigned queries.")])
    return [*items, msg]


async def oa_doc_responder(call):
    """Scripted document analyst: one file_search call (with results) and file_citation annotations."""
    await asyncio.sleep(LATENCY["model"])
    brief = brief_from_text(brief_text_of(call))
    results = [FileSearchResult(file_id=FILE_IDS[d], filename=f"{d}.txt", score=0.9, text=CORPUS[d]["text"],
                                attributes={"source_id": d, "published": CORPUS[d]["published"]}) for d in brief.documents]
    fs = ResponseFileSearchToolCall(id=f"fs_{brief.task_id}", type="file_search_call", status="completed",
                                    queries=[brief.objective], results=results)
    answer, annotations = "", []
    for d in brief.documents:
        for sentence, f in CORPUS[d]["sentences"]:
            if f:
                answer += f["statement"] + "."
                annotations.append(AnnotationFileCitation(type="file_citation", file_id=FILE_IDS[d], filename=f"{d}.txt", index=len(answer)))
                answer += " "
    msg = ResponseOutputMessage(id=f"msg_{brief.task_id}", type="message", role="assistant", status="completed",
                                content=[ResponseOutputText(type="output_text", text=answer, annotations=annotations)])
    return [fs, msg]


def oa_fields_responder(call):
    return [assistant_message(json.dumps(fields_from_gold(json.loads(tag(brief_text_of(call), "items")))))]


def scripted(responder, n: int = 12) -> ScriptedModel:
    return ScriptedModel([ModelStep.respond(responder) for _ in range(n)])


print("scripted OpenAI responders ready")

In [ ]:
def best_chunk(results: list, statement: str):
    """The retrieved chunk that best supports a statement: shared numbers first, then shared words, then score."""
    if not results:
        return None
    words = set(re.findall(r"[a-z]{4,}", statement.lower()))
    return max(results, key=lambda r: (len(nums(r.text or "") & nums(statement)),
                                       len(words & set(re.findall(r"[a-z]{4,}", (r.text or "").lower()))), r.score or 0))


def drafts_from_annotations(items) -> tuple[list[dict], list[tuple[str, str]]]:
    """Nested-run items -> draft claims (from annotations) and a search log (from the hosted tool calls)."""
    drafts, log, chunks = [], [], defaultdict(list)
    annotations = [a for it in items if it.type == "message_output_item"
                   for part in it.raw_item.content if part.type == "output_text" for a in part.annotations]
    cited_urls = any(a.type == "url_citation" for a in annotations)
    cited_files = any(a.type == "file_citation" for a in annotations)
    for it in items:
        raw = it.raw_item
        kind = getattr(raw, "type", "")
        if kind == "web_search_call":
            action = raw.action
            if getattr(action, "type", "") != "search":
                continue                                   # open_page / find_in_page: not a search attempt
            q = getattr(action, "query", None) or "; ".join(getattr(action, "queries", None) or []) or "(query not reported)"
            sources = getattr(action, "sources", None)     # None when web_search_call.action.sources was not included
            if raw.status in ("failed", "incomplete"):
                outcome = "unavailable"                     # the hosted call reports no failure reason
            elif sources is None:
                outcome = "hits" if cited_urls else "empty" # sources not included: fall back to the answer's citations
            else:
                outcome = "hits" if sources else "empty"
            log.append((q, outcome))
        elif kind == "file_search_call":
            q = "file_search: " + ("; ".join(raw.queries or []) or "(query not reported)")
            if raw.status != "completed":                   # failed / incomplete: a gap, never an empty result
                log.append((q, "unavailable"))
                continue
            if raw.results is None:                         # results not included: fall back to the answer's citations
                log.append((q, "hits" if cited_files else "empty"))
                continue
            for res in raw.results:
                chunks[res.file_id].append(res)             # keep EVERY chunk; a file can return several
            log.append((q, "hits" if raw.results else "empty"))
            log += [(d, "hits") for d in dict.fromkeys(f"document {FILE_TO_ID.get(r.file_id, r.file_id)}" for r in raw.results)]
        elif it.type == "message_output_item":
            for part in raw.content:
                if part.type != "output_text":
                    continue
                for a in part.annotations:
                    if a.type == "url_citation":
                        span = part.text[a.start_index:a.end_index].strip()
                        drafts.append({"statement": span.rstrip("."), "evidence": [Evidence(
                            source_id=URL_TO_ID.get(a.url, a.url), excerpt=span,
                            location=f"answer chars {a.start_index}-{a.end_index}", verbatim=False)]})
                at_index: dict[int, list] = defaultdict(list)  # several citations can share one insertion point
                for a in part.annotations:
                    if a.type == "file_citation":
                        at_index[a.index].append(a)
                prev = 0
                for index in sorted(at_index):
                    stmt, prev = part.text[prev:index].strip(), index
                    if not stmt:
                        continue
                    evidence = []
                    for a in at_index[index]:
                        res = best_chunk(chunks.get(a.file_id, []), stmt)
                        sid = (res.attributes or {}).get("source_id") if res else FILE_TO_ID.get(a.file_id, a.file_id)
                        ev = Evidence(source_id=sid, excerpt=res.text if res else stmt, location=f"file {a.filename}",
                                      verbatim=res is not None)
                        if ev not in evidence:
                            evidence.append(ev)
                    drafts.append({"statement": stmt.rstrip("."), "evidence": evidence})
    return drafts, log


def make_structurer(model) -> Agent:
    return Agent(name="Field extractor", instructions=FIELDS_SYSTEM, model=model, output_type=ClaimFieldsBatch)


async def structure_openai(structurer: Agent, brief: Brief, drafts: list[dict]) -> list[Claim]:
    if not drafts:
        return []
    items = [{"ref": f"d{i}", "statement": d["statement"], "excerpt": d["evidence"][0].excerpt} for i, d in enumerate(drafts)]
    out = await Runner.run(structurer, f"Allowed facets: {brief.facets}\n<items>{json.dumps(items)}</items>", run_config=OFFLINE)
    fields = {f.ref: f for f in out.final_output_as(ClaimFieldsBatch).claims}
    return [Claim(claim_id=f"{brief.task_id}-{i}", statement=d["statement"], evidence=d["evidence"],
                  **fields[f"d{i}"].model_dump(exclude={"ref"})) for i, d in enumerate(drafts) if f"d{i}" in fields]


def make_extractor(structurer: Agent):
    async def extract(result: RunResult) -> str:            # custom_output_extractor: must be async, returns str
        brief = brief_from_text(result.input if isinstance(result.input, str) else result.input[-1]["content"])
        drafts, log = drafts_from_annotations(result.new_items)
        return build_envelope(brief, await structure_openai(structurer, brief, drafts), log).model_dump_json()
    return extract


def failure_to_envelope(ctx, error: Exception) -> str:      # failure_error_function: ctx is a ToolContext
    brief = Brief.model_validate_json(ctx.tool_arguments)
    ftype, retryable = (("timeout", True) if isinstance(error, openai.APITimeoutError) else
                        ("rate_limited", True) if isinstance(error, openai.RateLimitError) else
                        ("permission_denied", False) if isinstance(error, (openai.AuthenticationError, openai.PermissionDeniedError)) else
                        ("internal_error", False))
    return failed_envelope(brief, ftype, retryable, f"{type(error).__name__}: {error}").model_dump_json()


DELEGATION_TOOLS = {"research_web": "web_researcher", "analyze_documents": "document_analyst"}


class PlanGate(AgentHooks):
    """AC1 on OpenAI: check the manager's COMPLETE plan before any specialist runs.

    on_llm_end fires right after the model returns and before the SDK executes the tool calls, so a PlanError
    raised here stops the run with no worker started, exactly like check_plan() before fan_out() on Claude.
    """

    def __init__(self):
        self.briefs: list[Brief] | None = None

    async def on_llm_end(self, context, agent, response) -> None:
        calls = [i for i in response.output if getattr(i, "type", "") == "function_call" and i.name in DELEGATION_TOOLS]
        if not calls or self.briefs is not None:      # later turns (retries the manager asks for) are not a new plan
            return
        try:
            briefs = [Brief.model_validate_json(c.arguments) for c in calls]
        except ValidationError as exc:
            raise PlanError(f"a delegation call is not a valid Brief: {exc.errors()[0]['msg']}") from None
        for c, b in zip(calls, briefs):
            if DELEGATION_TOOLS[c.name] != b.subagent:
                raise PlanError(f"{b.task_id}: {c.name} cannot run a {b.subagent} brief")
        check_plan(briefs)
        self.briefs = briefs


def build_openai_team(web_model, doc_model, fields_model, manager_model, web_tools=(), doc_tools=(), *, gate: bool = True) -> Agent:
    structurer = make_structurer(fields_model)
    web = Agent(name="Web researcher", instructions=WEB_SYSTEM, model=web_model, tools=list(web_tools),
                model_settings=ModelSettings(response_include=["web_search_call.action.sources"]))
    docs = Agent(name="Document analyst", instructions=DOC_SYSTEM, model=doc_model, tools=list(doc_tools))
    common = dict(parameters=Brief, input_builder=lambda opts: render_brief(Brief.model_validate(opts["params"])),
                  custom_output_extractor=make_extractor(structurer), failure_error_function=failure_to_envelope,
                  run_config=OFFLINE, max_turns=6)
    return Agent(name="Research coordinator", instructions=COORDINATOR_SYSTEM, model=manager_model, hooks=PlanGate() if gate else None,
                 model_settings=ModelSettings(parallel_tool_calls=True), tools=[
                     web.as_tool(tool_name="research_web", tool_description=(
                         "Research one facet group on the public web. Returns SubagentResult JSON."), **common),
                     docs.as_tool(tool_name="analyze_documents", tool_description=(
                         "Analyze the attached reports for the given facets. Returns SubagentResult JSON."), **common)])


def scripted_manager() -> ScriptedModel:
    calls = [function_call("research_web" if t["subagent"] == "web_researcher" else "analyze_documents", t, call_id=f"call_{t['task_id']}")
             for t in PLAN]
    return ScriptedModel([calls, [assistant_message("All four briefs returned; handing over to synthesis.")]])


OA_MODELS: dict[str, ScriptedModel] = {}      # the latest offline team's scripted models (the acceptance checks read them)


def offline_team() -> Agent:
    OA_MODELS.update(web=scripted(oa_web_responder), docs=scripted(oa_doc_responder), manager=scripted_manager())
    return build_openai_team(OA_MODELS["web"], OA_MODELS["docs"], scripted(oa_fields_responder),
                             OA_MODELS["manager"], web_tools=[WebSearchTool()],
                             doc_tools=[FileSearchTool(vector_store_ids=["vs_offline"], include_search_results=True)])


def envelopes_of(result: RunResult) -> list[SubagentResult]:
    return [SubagentResult.model_validate_json(i.output) for i in result.new_items if i.type == "tool_call_output_item"]


timings = {}
for label, limit in [("sequential", 1), ("parallel", 4)]:
    cfg = RunConfig(tracing_disabled=True, tool_execution=ToolExecutionConfig(max_function_tool_concurrency=limit))
    t0 = time.perf_counter()
    oa_result = await Runner.run(offline_team(), QUESTION, run_config=cfg)
    timings[label] = time.perf_counter() - t0
print(f"OpenAI sequential {timings['sequential']:.2f}s | parallel {timings['parallel']:.2f}s | speed-up x{timings['sequential'] / timings['parallel']:.1f}")
oa_results = envelopes_of(oa_result)
for r in oa_results:
    print(f"  {r.task_id} {r.subagent:16} {r.status:8} claims={len(r.claims)} failed={r.failed_queries} empty={r.empty_queries}")

In [ ]:
# AC1 on OpenAI: a narrow plan is rejected by PlanGate before any specialist runs.
narrow_web = scripted(oa_web_responder)
narrow_team = build_openai_team(narrow_web, scripted(oa_doc_responder), scripted(oa_fields_responder),
                                ScriptedModel([[function_call("research_web", PLAN[0], call_id="c1")], [assistant_message("ok")]]))
try:
    await Runner.run(narrow_team, QUESTION, run_config=OFFLINE)
    print("narrow plan ran (unexpected)")
except PlanError as exc:
    print("narrow plan rejected before delegation:", exc, "| specialist model calls:", len(narrow_web.calls))

# The next two runs test ONE specialist on purpose, so they switch the plan gate off (gate=False).
# A nested run that raises (here: a model-call timeout) reaches the manager as a structured envelope.
timeout_error = openai.APITimeoutError(request=httpx2.Request("POST", "https://api.openai.com/v1/responses"))
crashing = build_openai_team(ScriptedModel([ModelStep.raise_error(timeout_error)]), scripted(oa_doc_responder),
                             scripted(oa_fields_responder),
                             ScriptedModel([[function_call("research_web", PLAN[1], call_id="c1")], [assistant_message("noted")]]),
                             gate=False)
crash_result = await Runner.run(crashing, QUESTION, run_config=OFFLINE)
print(envelopes_of(crash_result)[0].model_dump_json(indent=1, exclude={"claims"}))

# Explicit context reached the nested agent: the scripted model recorded what it was sent.
probe_web = scripted(oa_web_responder)
probe = build_openai_team(probe_web, scripted(oa_doc_responder), scripted(oa_fields_responder),
                          ScriptedModel([[function_call("research_web", PLAN[0], call_id="c1")], [assistant_message("ok")]]),
                          gate=False)
await Runner.run(probe, QUESTION, run_config=OFFLINE)
seen = brief_text_of(probe_web.first_call)
print("nested web researcher received:", brief_lint(seen) or "a complete brief", "|", seen[:60].replace("\n", " "))

In [ ]:
# OpenAI recovery round: the same policy, the retry run directly from code with the same extractor.
async def recover_openai(results: list[SubagentResult], retries: int = 1):
    web = Agent(name="Web researcher", instructions=WEB_SYSTEM, model=scripted(oa_web_responder), tools=[WebSearchTool()])
    extract = make_extractor(make_structurer(scripted(oa_fields_responder)))
    final, decisions = list(results), []
    for i, r0 in enumerate(results):
        b = Brief(**next(t for t in PLAN if t["task_id"] == r0.task_id))
        left = retries
        while (action := decide(final[i], left)) == "retry_modified":
            alt = next(a for a in final[i].alternatives if a.startswith("query: "))[len("query: "):]
            retry = b.model_copy(update={"task_id": f"{b.task_id}-R{retries - left + 1}", "queries": [alt]})
            nested = await Runner.run(web, render_brief(retry, final[i].claims, final[i].failed_queries), run_config=OFFLINE)
            new = SubagentResult.model_validate_json(await extract(nested))
            decisions.append({"task": b.task_id, "action": "retry_modified", "retry": retry.task_id, "outcome": new.status})
            final[i], left = merge_envelopes(final[i], new), left - 1
        decisions.append({"task": b.task_id, "action": action, "status": final[i].status})
    return final, decisions


oa_results, oa_decisions = await recover_openai(oa_results)
print([d for d in oa_decisions if d["task"] == "T2"])


async def synthesize_openai(claims, pairs, levels, rows) -> str:
    synth = Agent(name="Synthesizer", instructions=SYNTH_SYSTEM,
                  model=scripted(lambda call: [assistant_message(scripted_summary(brief_text_of(call)))], 1))
    return (await Runner.run(synth, synthesis_prompt(claims, pairs, levels, rows), run_config=OFFLINE)).final_output


oa_claims = merge_claims(oa_results)
oa_pairs = classify_all(oa_claims)
oa_levels = support_levels(oa_claims, oa_pairs)
oa_rows = coverage_rows([Brief(**t) for t in PLAN], oa_results)
oa_records = reconcile(conflict_records(oa_claims, oa_pairs))
oa_summary = await synthesize_openai(oa_claims, oa_pairs, oa_levels, oa_rows)
report_openai = render_report("OpenAI Agents SDK", oa_claims, oa_pairs, oa_levels, oa_rows, oa_records, oa_summary)
print("audit (OpenAI build):", audit(oa_summary, oa_claims, oa_pairs, oa_rows) or "PASS")
print("same claim IDs and statements as the Claude build:",
      [(c.claim_id, c.statement) for c in oa_claims] == [(c.claim_id, c.statement) for c in claims])
for c in oa_claims[:2] + [c for c in oa_claims if c.facet == "safety"][:1]:
    e = c.evidence[0]
    print(f"  {c.claim_id} <- {e.source_id} | {e.location} | verbatim={e.verbatim} | {e.excerpt[:70]!r}")

> **Observe:** the two builds produce the same claims, conflicts and coverage, but not the same **evidence**. Claude's evidence is the exact source sentence with offsets or block indices. OpenAI's web evidence is a span of the model's own answer plus a URL, and its document evidence is a whole retrieved chunk. Both are better than prompt-only citations, but they are not equally checkable: Step 9 has to fetch the source to verify a `verbatim=False` excerpt.

> **Try it:** in `build_openai_team`, change `parallel_tool_calls=True` to `False`, or set `max_function_tool_concurrency=1` in the run config, and compare the timing.

## Step 9 — Evaluation and comparison

The metrics from the README's [evaluation plan](README.md#evaluation-plan), computed by code against the gold corpus, for the Claude build, the OpenAI build and a deliberately **lossy baseline**: an intermediate summarizer, a synthesizer that picks or averages conflicting values, and no coverage or contested sections. If the suite did not score the baseline clearly worse, it would not be testing provenance.

In [ ]:
def evidence_verified(c: Claim) -> bool:
    for e in c.evidence:
        text = CORPUS.get(e.source_id, {}).get("text")
        if text is None:
            return False
        if e.verbatim and e.excerpt not in text:
            return False
        if not e.verbatim and c.value is not None and c.value not in nums(text):    # fetch the page, check the number
            return False
    return True


def section(report: str, title: str) -> str:
    m = re.search(rf"^## {re.escape(title)}\n(.*?)(?=^## |\Z)", report, re.S | re.M)
    return m.group(1) if m else ""


def evaluate(stack: str, summary: str, report: str, claims, pairs, rows, records, produced: list[Claim]) -> dict:
    by_id = {c.claim_id: c for c in claims}
    sentences = split_sentences(summary)
    cite_pairs = [(s, cid) for s in sentences for cid in re.findall(r"\[(C\d+)\]", s)]
    supported = 0
    for s, cid in cite_pairs:
        c = by_id.get(cid)
        cited = [by_id[x] for x in re.findall(r"\[(C\d+)\]", s) if x in by_id]
        allowed = set().union(*(evidence_numbers(x) for x in cited))      # the cited evidence, not the whole source
        supported += bool(c) and evidence_verified(c) and nums(REF.sub("", s)) <= allowed   # every number traced
    not_full = [r for r in rows if r["label"] != "FULL"]
    cov = section(report, "Coverage and limitations")
    contested = section(report, "Contested findings")
    srcs = section(report, "Sources")
    preserved = sum(all(fmt(s["value"]) in contested and all(f"[{x}]" in contested for x in s["sources"])
                        and all(p in contested for p in s["published"]) and all(m in contested for m in s["methods"])
                        for s in r["sides"]) for r in records)
    ef_total = sum(len(r["empty"]) + len(r["failed"]) for r in rows)
    ef_ok = sum(q in cov for r in rows for q in r["empty"] + r["failed"]) - (10 if re.search(r"\bno evidence\b", report, re.I) else 0)
    return {
        "stack": stack,
        "citation precision": round(supported / len(cite_pairs), 2) if cite_pairs else 0.0,
        "sentence citation coverage": round(sum(bool(REF.search(s)) for s in sentences) / len(sentences), 2),
        "facet coverage": f"{sum(r['label'] != 'NONE' for r in rows)}/{len(rows)}",
        "gap annotation recall": round(sum(f"[gap:{r['facet']}]" in summary and r["reason"] in cov for r in not_full) / len(not_full), 2) if not_full else 1.0,
        "empty vs failed accuracy": round(max(ef_ok, 0) / ef_total, 2) if ef_total else 1.0,
        "conflict preservation": round(preserved / len(records), 2) if records else 1.0,
        "attribution survival": round(sum(all(f"[{e.source_id}]" in srcs for e in c.evidence) for c in produced) / len(produced), 2),
        "audit problems": len(audit(summary, claims, pairs, rows)),
    }


def cid(prefix: str, cl=claims) -> str:
    return next(c.claim_id for c in cl if c.statement.startswith(prefix))


LOSSY_SUMMARY = (f"Riverton's program carried 5.6 million trips in 2024 [{cid('Operators recorded 5.6')}]. "
                 f"The fleet avoided about 2,650 tonnes of CO2 in 2024 [{cid('Including manufacturing')}]. "
                 f"Injury rates were low at 21 injury collisions per million trips [{cid('Police recorded 21')}]. "
                 "There is no evidence that the program generated public revenue. "
                 f"Shared scooters are limited to 20 km/h in the city centre [{cid('Shared scooters are limited')}].")
report_lossy = render_report("lossy baseline", claims, pairs, levels, rows, records, LOSSY_SUMMARY, lossy=True)

produced_claude = [c for r in claude_run["results"] for c in r.claims]
produced_openai = [c for r in oa_results for c in r.claims]
scores = [evaluate("Claude", summary, report_claude, claims, pairs, rows, records, produced_claude),
          evaluate("OpenAI", oa_summary, report_openai, oa_claims, oa_pairs, oa_rows, oa_records, produced_openai),
          evaluate("lossy baseline", LOSSY_SUMMARY, report_lossy, claims, pairs, rows, records, produced_claude)]
scores[0]["parallel speed-up"], scores[1]["parallel speed-up"], scores[2]["parallel speed-up"] = (
    round(t_seq / t_par, 1), round(timings["sequential"] / timings["parallel"], 1), "-")
keys = list(scores[0])
table = ["| Metric | " + " | ".join(str(s["stack"]) for s in scores) + " |", "|---|" + "---|" * len(scores)]
table += [f"| {k} | " + " | ".join(str(s[k]) for s in scores) + " |" for k in keys[1:]]
display(Markdown("\n".join(table)))
print("lossy audit:", *audit(LOSSY_SUMMARY, claims, pairs, rows), sep="\n  ")

In [ ]:
# Acceptance criteria from the README (AC1-AC10), checked on what each build actually produced and rendered.
def claude_prompt_of(body: dict) -> str:
    first = body["messages"][0]["content"]
    return first if isinstance(first, str) else next(b["text"] for b in reversed(first) if b.get("type") == "text")


def plan_gate_ok(briefs: list[Brief]) -> bool:
    """AC1: the plan the run used passes check_plan, and the same plan minus one facet is rejected."""
    try:
        check_plan(briefs)
    except PlanError:
        return False
    narrow = [b for b in briefs if "economy" not in b.facets]
    try:
        check_plan(narrow)
        return False
    except PlanError:
        return True


def acceptance(briefs, worker_prompts, speed_up, results_in_one_message, run_results,
               cl, prs, lv, rws, recs, smry, rep, score) -> dict[str, bool]:
    t2r = next(r for r in run_results if r.task_id == "T2")
    ridership_2024 = [c for c in cl if c.metric == "trips per year" and c.period == "2024" and c.value in (4.1, 4.0)]
    derivative = next(c for c in cl if any(e.source_id == "W4" for e in c.evidence))
    return {
        "AC1 full plan accepted, narrow plan rejected": plan_gate_ok(briefs),
        "AC2 every subagent prompt is a complete brief": bool(worker_prompts) and all(not brief_lint(p) for p in worker_prompts),
        "AC3 one response, concurrent workers, results in one message": speed_up >= 2.0 and results_in_one_message,
        "AC4 every claim has registered evidence": all(c.evidence and all(e.source_id in REGISTRY for e in c.evidence) for c in cl),
        "AC5 timeout -> partial envelope, run continues": t2r.status == "partial" and bool(t2r.claims) and bool(t2r.failed_queries)
                                                          and t2r.failure_type in ("timeout", "unavailable"),   # hosted search reports no reason
        "AC6 empty is not failed": "Riverton scooter enforcement fines 2024" in section(rep, "Coverage and limitations")
                                   and not re.search(r"\bno evidence\b", rep, re.I),
        "AC7 contested section shows both sides of every record": score["conflict preservation"] == 1.0 and len(recs) >= 3,
        "AC8 audit passes": audit(smry, cl, prs, rws) == [],
        "AC9 citation precision >= 0.95": score["citation precision"] >= 0.95,
        "AC10 origins, not URLs": all(lv[c.claim_id] == "well_established" for c in ridership_2024)
                                  and origins(derivative) == {"W1"},
    }


claude_prompts = [claude_prompt_of(b) for b in sc.requests if route(b) in ("web_researcher", "document_analyst")]
openai_prompts = [brief_text_of(c) for m in (OA_MODELS["web"], OA_MODELS["docs"]) for c in m.calls]
second_manager_call = OA_MODELS["manager"].calls[1].input
oa_outputs = sum((i.get("type") if isinstance(i, dict) else getattr(i, "type", None)) == "function_call_output"
                 for i in second_manager_call)
checks = {
    "Claude": (claude_run["briefs"], claude_prompts, t_seq / t_par,
               sum(b["type"] == "tool_result" for b in last_coord["content"]) == len(claude_run["briefs"]),
               claude_run["results"], claims, pairs, levels, rows, records, summary, report_claude, scores[0]),
    "OpenAI": ([Brief(**t) for t in PLAN], openai_prompts, timings["sequential"] / timings["parallel"], oa_outputs == len(PLAN),
               oa_results, oa_claims, oa_pairs, oa_levels, oa_rows, oa_records, oa_summary, report_openai, scores[1]),
}
for name, args in checks.items():
    res = acceptance(*args)
    print(name, "PASS" if all(res.values()) else "FAIL", {k: v for k, v in res.items() if not v} or "")

> **Observe:** both builds meet every threshold, and the lossy baseline fails on exactly the metrics it should: citation precision (an averaged number nobody published), sentence coverage (an unreferenced claim), gap annotation and empty-vs-failed (the outage became "no evidence"), conflict preservation and attribution survival. The acceptance cell checks all ten criteria on what each build actually produced: the plan and its narrow variant (AC1), the prompts the workers really received (AC2), the measured speed-up and the single results message (AC3), and the **rendered** report for conflicts and citation precision (AC7, AC9). A report that kept the records but dropped the contested section, or a summary with no citations, fails.

**Write your comparison.** Using the table and the [comparison section](README.md#claude-vs-openai-comparison), write five sentences for Riverton's policy unit: which stack you would choose for them and why, what you would keep vendor-neutral, and what you would verify in a live run first. You will score it in the homework rubric.

## Step 10 — Live runs (opt-in)

Three optional cells. Each one is skipped unless its flag is `True` in the setup cell.

1. **(live) Claude pipeline** on the real Messages API: the same code with a real `AsyncAnthropic` client. The search backend and documents are still the fixture corpus (so citations are checkable), but the coordinator, workers, field extraction and synthesizer are real model calls. Expect a different plan and wording; the plan check, envelopes, audit and eval still apply. Roughly $0.05–0.25 (estimate; the cell prints the token counts).
2. **(live) OpenAI `file_search`**: uploads `D1`–`D4` to a temporary vector store with `source_id` and `published` attributes, runs the real document analyst with `FileSearchTool(include_search_results=True)`, converts its `file_citation` annotations with the same extractor, then deletes the files and the store. Well under $0.05 (estimate: `gpt-6-luna` tokens plus $2.50 per 1,000 `file_search` calls; storage under 1 GB is free).
3. **Claude Agent SDK variant**: the configuration is always built and checked offline; the run itself needs `RUN_AGENT_SDK = True` and spawns Claude Code under your own login or key. In this variant a **model** carries the records from the subagents to the synthesizer, so your code cannot merge, classify, render or audit in between. Two consequences: the claim contract asks each subagent to number its own claims (`T1-1`, `T1-2`, ...) so the synthesizer has IDs to cite, and the synthesizer gets its own prompt written for raw subagent JSON. For an audited brief, take the subagents' JSON out of the run and pass it through the same downstream code as Stack A (merge, classify, render, audit).

In [ ]:
# (live) Claude: the whole pipeline against the real API. Skipped unless RUN_LIVE is True.
if RUN_LIVE:
    USAGE.clear()
    live = anthropic.AsyncAnthropic()                  # reads ANTHROPIC_API_KEY
    try:
        live_run = await coordinate_claude(live)
        lc = merge_claims(live_run["results"])
        lp = classify_all(lc)
        lv, lr = support_levels(lc, lp), coverage_rows(live_run["briefs"], live_run["results"])
        lrec = reconcile(conflict_records(lc, lp))
        ls = await synthesize_claude(live, lc, lp, lv, lr)
        live_report = render_report(f"Claude live ({CLAUDE_MODEL})", lc, lp, lv, lr, lrec, ls)
        display(Markdown(live_report))
        print("audit:", audit(ls, lc, lp, lr) or "PASS")
        print(evaluate("Claude live", ls, live_report, lc, lp, lr, lrec, [c for r in live_run["results"] for c in r.claims]))
    except PlanError as exc:
        print("The live coordinator's plan was rejected by check_plan (R1 working as intended):", exc)
    tin, tout = sum(u[1] for u in USAGE), sum(u[2] for u in USAGE)
    print(f"{len(USAGE)} calls, {tin} input / {tout} output tokens; check the price per million tokens for {CLAUDE_MODEL}.")
else:
    print("Skipped (RUN_LIVE = False).")

In [ ]:
# (live) OpenAI file_search: real vector store, real file_citation annotations, same extractor. Skipped unless RUN_LIVE.
if RUN_LIVE:
    oc = openai.OpenAI()
    vs = oc.vector_stores.create(name="capstone02-riverton", expires_after={"anchor": "last_active_at", "days": 1})
    uploaded = []
    try:
        for d in DOC_IDS:
            f = oc.vector_stores.files.upload_and_poll(
                vector_store_id=vs.id, file=(f"{d}.txt", CORPUS[d]["text"].encode()),
                attributes={"source_id": d, "published": CORPUS[d]["published"]})
            uploaded.append(f.id)
            FILE_TO_ID[f.id] = d
        analyst = Agent(name="Document analyst", instructions=DOC_SYSTEM, model=OPENAI_MODEL,
                        tools=[FileSearchTool(vector_store_ids=[vs.id], include_search_results=True)])
        brief = Brief(**PLAN[3])
        nested = await Runner.run(analyst, render_brief(brief))
        drafts, log = drafts_from_annotations(nested.new_items)
        live_claims = await structure_openai(make_structurer(OPENAI_MODEL), brief, drafts)
        env = build_envelope(brief, live_claims, log)
        print(env.status, len(env.claims), "claims")
        for c in env.claims:
            print(f"  {c.statement} | {c.metric}={c.value} ({c.period}) <- {c.evidence[0].source_id} {c.evidence[0].location}")
    finally:
        for fid in uploaded:
            oc.files.delete(fid)
        oc.vector_stores.delete(vs.id)
        print("vector store and files deleted")
else:
    print("Skipped (RUN_LIVE = False).")

In [ ]:
# Claude Agent SDK variant: the configuration is built and checked offline; nothing starts unless RUN_AGENT_SDK.
from claude_agent_sdk import AgentDefinition, ClaudeAgentOptions, create_sdk_mcp_server, tool


@tool("search_web", "Search the public web index for sources about Riverton. One query per call.", {"query": str})
async def sdk_search_web(args: dict) -> dict:
    sources, error = await search_with_recovery(args["query"])
    if error:
        return {"content": [{"type": "text", "text": json.dumps(error)}], "is_error": True}
    payload = [{"source_id": s["source_id"], "url": s["url"], "title": s["title"], "published": s["published"],
                "sentences": [t for t, _ in s["sentences"]]} for s in sources]
    return {"content": [{"type": "text", "text": json.dumps(payload) if payload else "No results found."}]}


CLAIM_CONTRACT = ("End with ONE JSON object: {\"task_id\", \"status\": \"success|empty|partial|failed\", \"failure_type\": null|\"timeout\"|..., "
                  "\"facets\": [...], \"attempted\": [...], \"failed_queries\": [...], \"empty_queries\": [...], \"claims\": [{\"claim_id\" "
                  "(your task id, a dash and a number: T1-1, T1-2, ...), \"facet\", \"statement\", \"source_id\", "
                  "\"excerpt\" (copied verbatim), \"published\", \"period\"}], \"alternatives\": [...]}. "
                  "A failed search is 'partial' or 'failed', never 'empty'.")
# The synthesizer here reads raw subagent JSON (no code-assigned C# IDs, no pair table, no coverage table),
# so it gets a prompt written for exactly that input instead of SYNTH_SYSTEM.
SDK_SYNTH_SYSTEM = """You write the summary section of a research brief from the subagent results (JSON) in your prompt.
Rules:
- 6 to 12 sentences. Every sentence ends with the claim_id values it uses, e.g. [T1-2][T4-1], or with a gap tag [gap:<facet>].
- Use only claims in those results; copy numbers exactly; keep qualifiers such as 'estimated'.
- When two claims give different values for the same thing and period, give both values in one sentence with both IDs. Never pick one.
- For every facet of a result whose status is partial or failed, write one sentence with its gap tag saying what is missing and why.
- A failed source is a gap, never 'no evidence'. A search with zero results is a finding."""
AGENTS = {
    "web_researcher": AgentDefinition(
        description="Searches the web index for ONE assigned facet group and returns claim records.",
        prompt="You research only the facets in your brief with mcp__corpus__search_web. " + CLAIM_CONTRACT,
        tools=["mcp__corpus__search_web"]),
    "document_analyst": AgentDefinition(
        description="Reads the attached reports in the working directory for the assigned facets.",
        prompt="You read only the report files named in your brief with Read. Report conflicting values, do not choose. " + CLAIM_CONTRACT,
        tools=["Read"]),
    "synthesizer": AgentDefinition(
        description="Writes summary sentences from claim records passed in its prompt.",
        prompt=SDK_SYNTH_SYSTEM, tools=[],                # intent: no tools. The docs define only "omitted = inherit all"
        disallowedTools=["mcp__corpus", "WebSearch", "WebFetch", "Agent"]),   # and "list = only those"; this documented deny list is the guarantee
}
SDK_COORDINATOR = (COORDINATOR_SYSTEM + "\nSpawn web_researcher and document_analyst subagents with the Agent tool, all "
                   "independent ones in the same response. Put the full brief (goal, facets, out-of-scope facets, queries "
                   "or file names, its task id T1..Tn, output contract) in each prompt. Then pass ALL returned JSON verbatim to the synthesizer.")
sdk_options = ClaudeAgentOptions(
    system_prompt=SDK_COORDINATOR, agents=AGENTS, model=CLAUDE_MODEL,
    allowed_tools=["Agent", "Read", "mcp__corpus__search_web"],       # "Agent" is the guide's "Task"
    mcp_servers={"corpus": create_sdk_mcp_server(name="corpus", tools=[sdk_search_web])},
    max_budget_usd=1.00, max_turns=30, env={"CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1"})

assert "Agent" in sdk_options.allowed_tools, "the coordinator must be able to spawn subagents"
assert not set(AGENTS["synthesizer"].tools), "the synthesizer must not search"
assert {"mcp__corpus", "WebSearch", "WebFetch", "Agent"} <= set(AGENTS["synthesizer"].disallowedTools), "documented deny list"
assert '"claim_id"' in CLAIM_CONTRACT and "[T1-2]" in SDK_SYNTH_SYSTEM, "the synthesizer can only cite IDs the subagents produce"
assert AGENTS["web_researcher"].tools == ["mcp__corpus__search_web"] and AGENTS["document_analyst"].tools == ["Read"]
print("Agent SDK configuration checked offline:", {k: v.tools for k, v in AGENTS.items()})

if RUN_AGENT_SDK:
    from claude_agent_sdk import AssistantMessage, ResultMessage, ToolUseBlock, query
    workdir = Path(tempfile.mkdtemp(prefix="riverton-"))
    for d in DOC_IDS:
        (workdir / f"{d}.txt").write_text(f"{CORPUS[d]['title']} ({CORPUS[d]['publisher']}, published {CORPUS[d]['published']})\n\n{CORPUS[d]['text']}")
    sdk_options.cwd = str(workdir)
    prompt = QUESTION + " Attached reports (files in the working directory): " + ", ".join(f"{d}.txt" for d in DOC_IDS)
    async for message in query(prompt=prompt, options=sdk_options):
        if isinstance(message, AssistantMessage):
            spawns = [b for b in message.content if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task")]
            if spawns:   # several in ONE assistant message = parallel spawning
                print(f"{len(spawns)} subagent call(s) in one response:", [b.input.get("subagent_type") for b in spawns])
        if isinstance(message, ResultMessage):
            print(message.subtype, "| cost (USD):", message.total_cost_usd)
            print((message.result or "")[:3000])
else:
    print("Run skipped (RUN_AGENT_SDK = False).")

## Recap

- **R1/R2:** the coordinator plans; code rejects a plan that misses a facet. `render_brief()` is the only channel to a subagent, and the same function serves both stacks (as the Claude worker's message and as the OpenAI `input_builder`).
- **R3:** several delegation calls in **one** response → concurrent workers (`asyncio.gather` with a semaphore on Claude; `as_tool` calls with a concurrency cap on OpenAI) → all results back in **one** message. Measured, not assumed.
- **R4:** evidence comes from platform citations (`search_result_location`, `char_location`; `url_citation`, `file_citation`), then a separate structured-output step adds the fields. Claude returns exact source text; OpenAI returns a URL plus an answer span, or a file plus its chunk.
- **R5:** the tool retries locally; the worker wrapper always returns an envelope with status computed from the log; the coordinator's policy retries once with the specialist's alternative and the partial findings, then accepts partial.
- **R6/R7/R8:** coverage, conflicts, trends and support levels are computed in code from records; the model writes only referenced summary sentences; the audit and the eval suite catch the lossy baseline on every provenance metric.

Next: [homework](02_homework.ipynb), with auto-checked extensions (a new source-conflict rule, coverage annotations for a report), a quiz, the rubric and two design reviews.